<a href="https://colab.research.google.com/github/mbachant/Character-Complexity/blob/main/Text_Complexity_Calc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# Text Complexity Analysis
#
# Inputs:
#   UTF-8 text files
#   glyph_complexity_database.pkl
#   noto_font_table.pkl
#   unicode_to_notofonts_map.pkl
#
# Outputs:
#
  # text_complexity_by_font.csv
  # character_complexity_by_text_font.csv
  # english_representation_comparison.csv
  # english_representation_character_complexity.csv
#
#
# The glyph complexity database is authoritative.
# Each text is analyzed using the pre-calculated complexity
# of each character in each selected font.

import os
import unicodedata
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from google.colab import drive, files

In [32]:
# Install Noto fonts for each Colab runtime
!apt-get update -qq
!apt-get install -y fonts-noto -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [33]:
# Set Matplotlib to use Unicode-capable Noto fonts so that characters
# from different writing systems are displayed instead of missing-glyph
# rectangles.

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# Find installed Noto fonts
noto_fonts = [
    f
    for f in font_manager.findSystemFonts()
    if "Noto" in f
]

print(
    f"Noto font files found: {len(noto_fonts):,}"
)

for font_path in noto_fonts[:20]:
    print(font_path)

Noto font files found: 2,394
/usr/share/fonts/truetype/noto/NotoSans-MediumItalic.ttf
/usr/share/fonts/truetype/noto/NotoSansEthiopic-ExtraCondensedExtraLight.ttf
/usr/share/fonts/truetype/noto/NotoSerif-SemiCondensedExtraLightItalic.ttf
/usr/share/fonts/truetype/noto/NotoSansGujarati-SemiCondensedExtraLight.ttf
/usr/share/fonts/truetype/noto/NotoSerifTamil-ExtraBold.ttf
/usr/share/fonts/truetype/noto/NotoSerifEthiopic-ExtraBold.ttf
/usr/share/fonts/truetype/noto/NotoSerifTamilSlanted-CondensedExtraLight.ttf
/usr/share/fonts/truetype/noto/NotoSansHebrew-CondensedBlack.ttf
/usr/share/fonts/truetype/noto/NotoSansDisplay-SemiCondensedItalic.ttf
/usr/share/fonts/truetype/noto/NotoNaskhArabicUI-Regular.ttf
/usr/share/fonts/truetype/noto/NotoSerifDevanagari-CondensedMedium.ttf
/usr/share/fonts/truetype/noto/NotoSerif-SemiCondensed.ttf
/usr/share/fonts/truetype/noto/NotoSansOriyaUI-CondensedBlack.ttf
/usr/share/fonts/truetype/noto/NotoSerifMyanmar-ExtraCondensed.ttf
/usr/share/fonts/truetype/

In [34]:
# Choose input/output locations
#
# "drive"    = Google Drive
# "computer" = upload/download using your computer

input_source = "drive"
output_source = "drive"

base_dir = "/content/drive/MyDrive/Character Complexity"

text_directory_name = "udhr_all_txts"

glyph_database_filename = (
    "glyph_complexity_database.pkl"
)

font_table_filename = (
    "noto_font_table.pkl"
)

unicode_to_notofonts_map_filename = (
    "unicode_to_notofonts_map.pkl"
)


In [35]:
# Paths / Mount Google Drive if needed

if input_source == "drive" or output_source == "drive":

    drive.mount(
        "/content/drive",
        force_remount=False
    )


input_dir = (
    base_dir
    if input_source == "drive"
    else "/content"
)

output_dir = (
    base_dir
    if output_source == "drive"
    else "/content"


)


glyph_database_path = os.path.join(
    base_dir,
    glyph_database_filename
)

font_table_path = os.path.join(
    base_dir,
    font_table_filename
)

unicode_to_notofonts_map_path = os.path.join(
    base_dir,
    unicode_to_notofonts_map_filename
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# ============================================================
# LOAD + VERIFY GLYPH COMPLEXITY DATABASE
# ============================================================

if not os.path.isfile(
    glyph_database_path
):

    raise FileNotFoundError(
        f"Glyph complexity database not found:\n"
        f"{glyph_database_path}"
    )


glyph_complexity_database = pd.read_pickle(
    glyph_database_path
)


# ------------------------------------------------------------
# Verify required database columns
# ------------------------------------------------------------

required_columns = {
    "font_id",
    "font_name",
    "codepoint",
    "character",
    "complexity",
    "representation_system",
    "representation_standard",
    "representation"
}


missing_columns = (
    required_columns
    - set(glyph_complexity_database.columns)
)


if missing_columns:

    raise ValueError(
        "Glyph complexity database is missing "
        f"required columns: "
        f"{sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# Identify record types
# ------------------------------------------------------------

font_record_mask = (
    glyph_complexity_database[
        "font_id"
    ].notna()
)


font_records = int(
    font_record_mask.sum()
)


morse_records = int(
    (
        glyph_complexity_database[
            "representation_system"
        ]
        == "International Morse Code"
    ).sum()
)


ueb_records = int(
    (
        glyph_complexity_database[
            "representation_system"
        ]
        == "Unified English Braille"
    ).sum()
)


representation_records = (
    morse_records
    + ueb_records
)


# ------------------------------------------------------------
# Complexity-score counts
# ------------------------------------------------------------

complexity_values = int(
    glyph_complexity_database[
        "complexity"
    ].notna().sum()
)


missing_complexity_values = int(
    glyph_complexity_database[
        "complexity"
    ].isna().sum()
)


# ------------------------------------------------------------
# Report database contents
# ------------------------------------------------------------

print()
print("=" * 60)
print("GLYPH COMPLEXITY DATABASE")
print("=" * 60)

print()
print(
    f"Path: "
    f"{glyph_database_path}"
)

print()

print(
    f"Total database records: "
    f"{len(glyph_complexity_database):,}"
)

print()
print("RENDERED / MEASURED RECORDS")

print(
    f"  Font glyph records: "
    f"{font_records:,}"
)

print(
    f"  International Morse Code records: "
    f"{morse_records:,}"
)

print(
    f"  Unified English Braille records: "
    f"{ueb_records:,}"
)

print(
    f"  Total representation records: "
    f"{representation_records:,}"
)

print()
print("DATABASE CONTENT")

print(
    f"  Fonts: "
    f"{glyph_complexity_database['font_id'].nunique():,}"
)

print(
    f"  Unique source characters: "
    f"{glyph_complexity_database['codepoint'].nunique():,}"
)

print()
print("COMPLEXITY SCORES")

print(
    f"  Records with complexity: "
    f"{complexity_values:,}"
)

print(
    f"  Records without complexity: "
    f"{missing_complexity_values:,}"
)


GLYPH COMPLEXITY DATABASE

Path: /content/drive/MyDrive/Character Complexity/glyph_complexity_database.pkl

Total database records: 58,364

RENDERED / MEASURED RECORDS
  Font glyph records: 58,228
  International Morse Code records: 50
  Unified English Braille records: 86
  Total representation records: 136

DATABASE CONTENT
  Fonts: 8
  Unique source characters: 48,242

COMPLEXITY SCORES
  Records with complexity: 58,354
  Records without complexity: 10


In [37]:
# Check glyph_complexity_database

print("REPRESENTATION SYSTEMS IN GLYPH DATABASE:")
print()

print(
    glyph_complexity_database[
        "representation_system"
    ]
    .value_counts(
        dropna=False
    )
)

print()
print("Non-font representation records:")

display(
    glyph_complexity_database.loc[
        glyph_complexity_database[
            "representation_system"
        ].notna(),
        [
            "representation_system",
            "representation_standard",
            "character",
            "representation",
            "complexity"
        ]
    ].head(20)
)

REPRESENTATION SYSTEMS IN GLYPH DATABASE:

representation_system
NaN                         58228
Unified English Braille        86
International Morse Code       50
Name: count, dtype: int64

Non-font representation records:


,representation_system,representation_standard,character,representation,complexity
58228,International Morse Code,ITU-R M.1677-1,A,.-,0.136263
58229,International Morse Code,ITU-R M.1677-1,B,-...,0.182923
58230,International Morse Code,ITU-R M.1677-1,C,-.-.,0.183263
58231,International Morse Code,ITU-R M.1677-1,D,-..,0.162210
58232,International Morse Code,ITU-R M.1677-1,E,.,0.126500
58233,International Morse Code,ITU-R M.1677-1,F,..-.,0.188156
58234,International Morse Code,ITU-R M.1677-1,G,--.,0.163796
58235,International Morse Code,ITU-R M.1677-1,H,....,0.190789
58236,International Morse Code,ITU-R M.1677-1,I,..,0.151219
58237,International Morse Code,ITU-R M.1677-1,J,.---,0.189504


In [38]:
# Load font table

if not os.path.isfile(font_table_path):

    raise FileNotFoundError(
        f"Font table not found:\n"
        f"{font_table_path}"
    )


font_table = pd.read_pickle(
    font_table_path
)

unicode_to_notofonts_map = pd.read_pickle(
    unicode_to_notofonts_map_path
)


print()

print(
    f"Fonts in font table: "
    f"{len(font_table):,}"
)

print(
    f"Characters in Unicode to Notofonts Map: "
    f"{len(unicode_to_notofonts_map):,}"
)


Fonts in font table: 2,460
Characters in Unicode to Notofonts Map: 79,029


In [39]:
# Load UTF-8 text files

if input_source == "drive":

    text_dir = os.path.join(
        base_dir,
        text_directory_name
    )

    if not os.path.isdir(text_dir):

        raise FileNotFoundError(
            f"Text directory not found:\n{text_dir}"
        )

    text_files = sorted(
        os.path.join(
            text_dir,
            filename
        )
        for filename in os.listdir(text_dir)
        if filename.endswith(".txt")
    )


elif input_source == "computer":

    print(
        "Upload the UTF-8 text files to analyze."
    )

    uploaded_texts = files.upload()

    if not uploaded_texts:

        raise ValueError(
            "No text files were uploaded."
        )

    text_files = sorted(
        os.path.join(
            "/content",
            filename
        )
        for filename in uploaded_texts
    )


else:

    raise ValueError(
        "input_source must be 'drive' or 'computer'."
    )


print()
print("TEXT FILES FOUND")
print()

for filepath in text_files:

    print(
        f"  {os.path.basename(filepath)}"
    )

print()
print(
    f"Total text files: "
    f"{len(text_files):,}"
)


TEXT FILES FOUND

  abk.txt
  acu.txt
  afk.txt
  agr.txt
  aja.txt
  ako.txt
  aln.txt
  amc.txt
  ame.txt
  amr.txt
  arl.txt
  arm.txt
  aru.txt
  arz.txt
  ass.txt
  atj.txt
  aub.txt
  auv1.txt
  aym.txt
  bba.txt
  bci.txt
  bcy.txt
  bem.txt
  bkl.txt
  blg.txt
  blu.txt
  boa.txt
  bpr.txt
  bra.txt
  brt.txt
  bsq.txt
  btb.txt
  bzc.txt
  cab.txt
  cak1.txt
  cbr.txt
  cbs.txt
  cbt.txt
  cbu.txt
  ccx.txt
  ceb.txt
  chj.txt
  chn.txt
  cic.txt
  cjd.txt
  cjk.txt
  cln.txt
  cni.txt
  coi.txt
  cot.txt
  cpp.txt
  cpu.txt
  crs.txt
  csa.txt
  czc.txt
  dag.txt
  den.txt
  dga.txt
  dinka.txt
  dns.txt
  dum.txt
  dut.txt
  dyo.txt
  edo.txt
  eml.txt
  eng.txt
  esg.txt
  est.txt
  ewe.txt
  fae.txt
  fal.txt
  fin.txt
  fji.txt
  foa.txt
  fri.txt
  frl.txt
  frn.txt
  frn1.txt
  frn2.txt
  fum.txt
  fum1.txt
  gac1.txt
  gac2.txt
  gag.txt
  gax.txt
  gej.txt
  ger.txt
  gkp1.txt
  gli1.txt
  gln.txt
  gls.txt
  grk.txt
  gua.txt
  guc.txt
  gun.txt
  hak.txt
  hat.txt


In [40]:
# Reads every text file, verifies UTF-8/NFC status, counts characters,
# and automatically creates files_to_inspect.
#
# A file is flagged if:
#   - it cannot be decoded as UTF-8
#   - it is empty or contains only whitespace
#   - it contains fewer than number_of_characters
#
# Change number_of_characters to set the minimum text length.

number_of_characters = 1000


texts = {}

verification_results = []

files_to_inspect = []


for filepath in text_files:

    filename = os.path.basename(filepath)

    language = os.path.splitext(
        filename
    )[0]


    try:

        with open(
            filepath,
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()


    except UnicodeDecodeError as e:

        verification_results.append({

            "file": filename,

            "language": language,

            "UTF8": False,

            "NFC": None,

            "characters": None,

            "unique_characters": None,

            "status":
                f"UTF-8 ERROR: {e}"

        })


        files_to_inspect.append(
            filename
        )

        continue


    is_nfc = unicodedata.is_normalized(
        "NFC",
        text
    )


    text = unicodedata.normalize(
        "NFC",
        text
    )


    character_count = len(text)

    counts = Counter(text)

    unique_character_count = len(counts)


    # Flag empty or whitespace-only files

    if not text.strip():

        verification_results.append({

            "file": filename,

            "language": language,

            "UTF8": True,

            "NFC": is_nfc,

            "characters": character_count,

            "unique_characters": 0,

            "status":
                "ERROR: empty or whitespace only"

        })


        files_to_inspect.append(
            filename
        )

        continue


    # Flag files below the minimum character threshold

    if character_count < number_of_characters:

        verification_results.append({

            "file": filename,

            "language": language,

            "UTF8": True,

            "NFC": is_nfc,

            "characters": character_count,

            "unique_characters":
                unique_character_count,

            "status":
                f"WARNING: fewer than "
                f"{number_of_characters:,} characters"

        })


        files_to_inspect.append(
            filename
        )


    else:

        verification_results.append({

            "file": filename,

            "language": language,

            "UTF8": True,

            "NFC": is_nfc,

            "characters": character_count,

            "unique_characters":
                unique_character_count,

            "status":
                "Contains text"

        })


    texts[language] = text


verification_table = pd.DataFrame(
    verification_results
)


print()
print("TEXT VERIFICATION")
print()

display(
    verification_table
)


print()
print("=" * 80)
print("FILES TO INSPECT")
print("=" * 80)
print()


if files_to_inspect:

    for filename in files_to_inspect:

        print(
            f'"{filename}",'
        )

    print()

    print(
        f"Total files to inspect: "
        f"{len(files_to_inspect):,}"
    )

else:

    print(
        "No files require inspection."
    )


TEXT VERIFICATION



,file,language,UTF8,NFC,characters,unique_characters,status
0,abk.txt,abk,True,True,11446,61,Contains text
1,acu.txt,acu,True,True,14563,55,Contains text
2,afk.txt,afk,True,True,10485,58,Contains text
3,agr.txt,agr,True,True,15139,65,Contains text
4,aja.txt,aja,True,True,9959,81,Contains text
...,...,...,...,...,...,...,...
256,uzb.txt,uzb,True,True,12307,67,Contains text
257,vmw.txt,vmw,True,False,10050,71,Contains text
258,wee.txt,wee,True,True,10491,63,Contains text
259,wls.txt,wls,True,True,10245,57,Contains text



FILES TO INSPECT

No files require inspection.


In [41]:
# Loads the language-code key and translates filenames such as
# "eng.txt" into their full language names.
#
# languages.txt format:
#     code<TAB>language name
#
# Example:
#     eng    English


language_key_path = os.path.join(
    base_dir,
    "udhr_all_txts",
    "udhr",
    "languages.txt"
)


language_key = pd.read_csv(
    language_key_path,
    sep="\t",
    header=None,
    names=[
        "language_code",
        "language_name"
    ],
    dtype=str
)


language_key = (
    language_key
    .dropna(subset=["language_code"])
    .drop_duplicates("language_code")
)


# Add language name to verification table

verification_table["language_name"] = (
    verification_table["language"]
    .map(
        language_key.set_index(
            "language_code"
        )["language_name"]
    )
)


# Put language name next to the code

verification_table = verification_table[
    [
        "file",
        "language",
        "language_name",
        "UTF8",
        "NFC",
        "characters",
        "unique_characters",
        "status"
    ]
]


display(
    verification_table
)

,file,language,language_name,UTF8,NFC,characters,unique_characters,status
0,abk.txt,abk,Abkhaz,True,True,11446,61,Contains text
1,acu.txt,acu,Achuar-Shiwiar,True,True,14563,55,Contains text
2,afk.txt,afk,Afrikaans,True,True,10485,58,Contains text
3,agr.txt,agr,Aguaruna,True,True,15139,65,Contains text
4,aja.txt,aja,Adja,True,True,9959,81,Contains text
...,...,...,...,...,...,...,...,...
256,uzb.txt,uzb,Uzbek (Latin),True,True,12307,67,Contains text
257,vmw.txt,vmw,Makua (Mozambique),True,False,10050,71,Contains text
258,wee.txt,wee,Sorbian,True,True,10491,63,Contains text
259,wls.txt,wls,Cymraeg (Welsh),True,True,10245,57,Contains text


In [42]:
# Inspect files

# Add any files you want to inspect manually.
# These are added to the automatically flagged files_to_inspect.

files_to_inspect.extend([
    "hnd.txt",
    "por.txt"
])

# Remove duplicates while preserving order

files_to_inspect = list(
    dict.fromkeys(files_to_inspect)
)


beginning_lines = 5
ending_lines = 5


for filename in files_to_inspect:

    language = os.path.splitext(
        filename
    )[0]

    text = texts.get(language)


    print()
    print(f"{'─' * 80}")
    print(f"{filename}", end="")

    if text is None:

        print("  ⚠ NOT LOADED")
        continue


    lines = text.splitlines()


    print(
        f"  |  {len(text):,} characters"
        f"  |  {len(lines):,} lines"
    )


    print(f"{'─' * 80}")


    print(
        "\n".join(
            lines[:beginning_lines]
        )
    )


    if len(lines) > beginning_lines + ending_lines:

        print("\n   ⋮\n")


    print(
        "\n".join(
            lines[-ending_lines:]
        )
    )


────────────────────────────────────────────────────────────────────────────────
hnd.txt  ⚠ NOT LOADED

────────────────────────────────────────────────────────────────────────────────
por.txt  |  11,459 characters  |  130 lines
────────────────────────────────────────────────────────────────────────────────
Declaração Universal dos Direitos do Homem
Preâmbulo
Considerando que o reconhecimento da dignidade inerente a todos os membros da família humana e dos seus direitos iguais e inalienáveis constitui o fundamento da liberdade, da justiça e da paz no mundo;

Considerando que o desconhecimento e o desprezo dos direitos do homem conduziram a actos de barbárie que revoltam a consciência da Humanidade e que o advento de um mundo em que os seres humanos sejam livres de falar e de crer, libertos do terror e da miséria, foi proclamado como a mais alta inspiração do Homem;

   ⋮


Artigo 30°
Nenhuma disposição da presente Declaração pode ser interpretada de maneira a envolver para qualquer E

In [43]:
# Manually exclude files from analysis if necessary

excluded_files = [
    "hnd.txt"
]


for filename in excluded_files:

    language = os.path.splitext(
        filename
    )[0]

    texts.pop(
        language,
        None
    )

    text_files = [
        filepath
        for filepath in text_files
        if os.path.basename(filepath) != filename
    ]


print(
    f"Excluded: "
    f"{len(excluded_files):,} files"
)

for filename in excluded_files:

    print(
        f"  {filename}"
    )


print(
    f"Files remaining: "
    f"{len(texts):,}"
)

Excluded: 1 files
  hnd.txt
Files remaining: 261


In [44]:
# Builds the character inventory for every text.
# Stops with an error if any text fails.
# Adds the full language name from language_key.

character_inventories = {}

inventory_failures = []


language_names = (
    language_key
    .set_index("language_code")["language_name"]
)


for language, text in texts.items():

    try:

        language_name = language_names.get(
            language,
            "UNKNOWN"
        )


        counts = Counter(text)

        records = []


        for character, count in counts.items():

            codepoint = ord(character)


            records.append({

                "language":
                    language,

                "language_name":
                    language_name,

                "character":
                    character,

                "codepoint":
                    codepoint,

                "unicode":
                    f"U+{codepoint:04X}",

                "unicode_name":
                    unicodedata.name(
                        character,
                        "UNKNOWN"
                    ),

                "category":
                    unicodedata.category(
                        character
                    ),

                "count":
                    count,

                "is_whitespace":
                    character.isspace()

            })


        inventory = pd.DataFrame(
            records
        )


        inventory = (
            inventory
            .sort_values(
                "count",
                ascending=False
            )
            .reset_index(drop=True)
        )


        character_inventories[
            language
        ] = inventory


    except Exception as e:

        inventory_failures.append(
            (
                language,
                type(e).__name__,
                str(e)
            )
        )


if inventory_failures:

    print(
        f"CHARACTER INVENTORY FAILED — "
        f"{len(inventory_failures):,} text(s)"
    )

    for language, error_type, error in inventory_failures:

        print(
            f"  {language}: "
            f"{error_type}: {error}"
        )

    raise RuntimeError(
        "Character inventory construction failed."
    )

else:

    print(
        f"CHARACTER INVENTORY COMPLETE — "
        f"{len(character_inventories):,} texts"
    )

CHARACTER INVENTORY COMPLETE — 261 texts


In [45]:
# Provides a compact overview of the character composition of every text.
# Counts letters, numbers, punctuation, symbols, and whitespace separately.

inventory_summary = []


for language, inventory in character_inventories.items():

    total_characters = (
        inventory["count"].sum()
    )


    whitespace = (
        inventory.loc[
            inventory["is_whitespace"],
            "count"
        ].sum()
    )


    letters = (
        inventory.loc[
            inventory["category"].str.startswith("L"),
            "count"
        ].sum()
    )


    numbers = (
        inventory.loc[
            inventory["category"].str.startswith("N"),
            "count"
        ].sum()
    )


    punctuation = (
        inventory.loc[
            inventory["category"].str.startswith("P"),
            "count"
        ].sum()
    )


    symbols = (
        inventory.loc[
            inventory["category"].str.startswith("S"),
            "count"
        ].sum()
    )


    inventory_summary.append({

        "language":
            language,

        "language_name":
            inventory["language_name"].iloc[0],

        "unique_characters":
            len(inventory),

        "total_characters":
            total_characters,

        "letters":
            letters,

        "numbers":
            numbers,

        "punctuation":
            punctuation,

        "symbols":
            symbols,

        "whitespace":
            whitespace

    })


inventory_summary = (
    pd.DataFrame(
        inventory_summary
    )
    .sort_values(
        "language_name"
    )
    .reset_index(drop=True)
)


display(
    inventory_summary
)

,language,language_name,unique_characters,total_characters,letters,numbers,punctuation,symbols,whitespace
0,abk,Abkhaz,61,11446,9124,661,254,28,1379
1,atj,Achehnese,60,13072,10649,83,270,0,2070
2,acu,Achuar-Shiwiar,55,14563,12319,89,385,0,1770
3,aja,Adja,81,9959,7513,83,243,0,2109
4,gax,Afaan Oromo (Oromiffa),61,10781,8874,92,322,0,1493
...,...,...,...,...,...,...,...,...,...
256,ako,Wama,62,4790,3505,51,202,0,1028
257,guc,Wayuu,59,10855,8909,87,520,0,1339
258,tsw,Western Sotho/Tswana/Setswana,58,12475,9947,83,202,2,2241
259,ccx,Zhuang,63,10320,8464,38,301,0,1517


In [46]:
# Determine which Noto fonts can render every non-whitespace
# character in each text.
#
# Uses only unicode_to_notofonts_map.
# glyph_complexity_database is NOT used.


universal_fonts_by_language = {}

font_coverage_records = []


for language, inventory in character_inventories.items():

    language_name = language_names.get(
        language,
        "UNKNOWN"
    )


    non_whitespace = inventory[
        ~inventory["is_whitespace"]
    ]


    required_codepoints = set(
        non_whitespace["codepoint"]
    )


    if not required_codepoints:

        universal_fonts = set()

    else:

        universal_fonts = None


        for codepoint in required_codepoints:

            font_record = (
                unicode_to_notofonts_map.get(
                    codepoint
                )
            )


            if font_record is None:

                font_ids = set()

            elif isinstance(font_record, dict):

                font_ids = set(
                    font_record.get(
                        "font_ids",
                        []
                    )
                )

            else:

                font_ids = set(
                    font_record
                )


            if universal_fonts is None:

                universal_fonts = font_ids

            else:

                universal_fonts &= font_ids


            if not universal_fonts:

                break


        if universal_fonts is None:

            universal_fonts = set()


    universal_fonts_by_language[
        language
    ] = universal_fonts


    for font_id in sorted(
        universal_fonts
    ):

        font_coverage_records.append({

            "language":
                language,

            "language_name":
                language_name,

            "font_id":
                font_id

        })


font_coverage_table = pd.DataFrame(
    font_coverage_records,
    columns=[
        "language",
        "language_name",
        "font_id"
    ]
)


print(
    f"Texts checked: "
    f"{len(universal_fonts_by_language):,}"
)

print(
    f"Texts with at least one fully-rendering font: "
    f"{sum(bool(font_ids) for font_ids in universal_fonts_by_language.values()):,}"
)

print(
    f"Texts with zero fully-rendering fonts: "
    f"{sum(not font_ids for font_ids in universal_fonts_by_language.values()):,}"
)

print(
    f"Text × fully-rendering Noto font records: "
    f"{len(font_coverage_table):,}"
)

Texts checked: 261
Texts with at least one fully-rendering font: 259
Texts with zero fully-rendering fonts: 2
Text × fully-rendering Noto font records: 98,453


In [47]:
# Remove texts that have zero fully-rendering Noto fonts.

zero_coverage_languages = [
    language
    for language, font_ids
    in universal_fonts_by_language.items()
    if not font_ids
]


if zero_coverage_languages:

    print(
        f"Removing {len(zero_coverage_languages):,} "
        f"text(s) with zero font coverage:"
    )

    for language in zero_coverage_languages:

        print(
            f"  {language} — "
            f"{language_names.get(language, 'UNKNOWN')}"
        )


    # Remove from universal font results

    for language in zero_coverage_languages:

        universal_fonts_by_language.pop(
            language,
            None
        )


    # Remove from loaded texts

    for language in zero_coverage_languages:

        texts.pop(
            language,
            None
        )


    # Remove character inventories

    for language in zero_coverage_languages:

        character_inventories.pop(
            language,
            None
        )


    # Remove from active file list

    text_files = [
        filepath
        for filepath in text_files
        if os.path.splitext(
            os.path.basename(filepath)
        )[0]
        not in zero_coverage_languages
    ]


else:

    print(
        "No texts have zero font coverage."
    )


print(
    f"Texts remaining: {len(texts):,}"
)

print(
    f"Universal font records remaining: "
    f"{len(universal_fonts_by_language):,}"
)

Removing 2 text(s) with zero font coverage:
  arz — Arabic (Alarabia)
  kkn — Hankuko (Korean)
Texts remaining: 259
Universal font records remaining: 259


In [48]:
# Check if one Noto font can render every character in every text.
#
# Uses:
#   - universal_fonts_by_language
#   - unicode_to_notofonts_map
#   - font_table

nonempty_font_sets = [
    set(font_ids)
    for font_ids in universal_fonts_by_language.values()
    if font_ids
]


if nonempty_font_sets:

    common_universal_fonts = set.intersection(
        *nonempty_font_sets
    )

else:

    common_universal_fonts = set()


common_universal_fonts = sorted(
    common_universal_fonts
)


# Make sure font IDs have the same type for the lookup.

font_table_lookup = font_table.copy()

font_table_lookup["font_id"] = (
    font_table_lookup["font_id"]
    .astype(str)
)

common_font_ids_as_strings = [
    str(font_id)
    for font_id in common_universal_fonts
]


common_universal_font_table = (
    font_table_lookup[
        font_table_lookup["font_id"].isin(
            common_font_ids_as_strings
        )
    ][
        [
            "font_id",
            "font_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("font_id")
    .reset_index(drop=True)
)


print()

print(
    f"Texts checked: "
    f"{len(universal_fonts_by_language):,}"
)

print(
    f"Texts with at least one fully-rendering font: "
    f"{sum(bool(font_ids) for font_ids in universal_fonts_by_language.values()):,}"
)

print(
    f"Texts with zero fully-rendering fonts: "
    f"{sum(not font_ids for font_ids in universal_fonts_by_language.values()):,}"
)

print(
    f"Fonts that can render every character "
    f"in every text with coverage: "
    f"{len(common_universal_fonts):,}"
)

print()

print(
    "Common universal font IDs:"
)

print(
    common_universal_fonts
)

print()

print(
    "Common universal fonts:"
)

display(
    common_universal_font_table
)


Texts checked: 259
Texts with at least one fully-rendering font: 259
Texts with zero fully-rendering fonts: 0
Fonts that can render every character in every text with coverage: 0

Common universal font IDs:
[]

Common universal fonts:


,font_id,font_name


In [49]:
# Number of Noto fonts capable of fully rendering every
# non-whitespace character in each text.

font_coverage_summary = pd.DataFrame([
    {
        "language": language,
        "language_name": language_names.get(
            language,
            "UNKNOWN"
        ),
        "fully_rendering_fonts": len(font_ids)
    }
    for language, font_ids
    in universal_fonts_by_language.items()
])


font_coverage_summary = (
    font_coverage_summary
    .sort_values(
        "fully_rendering_fonts",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    font_coverage_summary
)

,language,language_name,fully_rendering_fonts
0,zuu,Zulu,407
1,lue,Luvale,407
2,toi,Tonga,406
3,tru1,Chuuk (Trukese),406
4,tgl,Filipino (Tagalog),406
...,...,...,...
254,dum,Gonja,324
255,bra,Bambara,324
256,jpn,Japanese (Nihongo),80
257,chn,Chinese (Mandarin),80


In [50]:
# For each text, calculate how many of its fully-rendering fonts
# are also capable of rendering each other text.
#
# This identifies texts whose usable font pool overlaps strongly
# with the usable font pools of other texts.

languages = list(
    universal_fonts_by_language.keys()
)


common_font_records = []


for language in languages:

    this_fonts = set(
        universal_fonts_by_language[language]
    )


    if not this_fonts:
        continue


    shared_counts = []


    for other_language in languages:

        if other_language == language:
            continue


        other_fonts = set(
            universal_fonts_by_language[
                other_language
            ]
        )


        shared_counts.append(
            len(
                this_fonts & other_fonts
            )
        )


    common_font_records.append({

        "language":
            language,

        "language_name":
            language_names.get(
                language,
                "UNKNOWN"
            ),

        "fully_rendering_fonts":
            len(this_fonts),

        "mean_shared_fonts":
            np.mean(shared_counts),

        "median_shared_fonts":
            np.median(shared_counts),

        "maximum_shared_fonts":
            max(shared_counts),

        "texts_with_shared_fonts":
            sum(
                count > 0
                for count in shared_counts
            )

    })


common_font_summary = (
    pd.DataFrame(
        common_font_records
    )
    .sort_values(
        [
            "mean_shared_fonts",
            "texts_with_shared_fonts"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "TOP 10 TEXTS WITH THE MOST FONTS IN COMMON"
)

display(
    common_font_summary.head(10)
)

TOP 10 TEXTS WITH THE MOST FONTS IN COMMON


,language,language_name,fully_rendering_fonts,mean_shared_fonts,median_shared_fonts,maximum_shared_fonts,texts_with_shared_fonts
0,lue,Luvale,407,380.023256,405.0,407,258
1,zuu,Zulu,407,380.023256,405.0,407,258
2,atj,Achehnese,406,380.019380,405.0,406,258
3,bcy,Bichelamar,406,380.019380,405.0,406,258
4,bem,Bemba,406,380.019380,405.0,406,258
5,bkl,Bikol/Bicolano,406,380.019380,405.0,406,258
6,blu,"Hmong (Miao), Sichuan-Guizhou-Yunnan",406,380.019380,405.0,406,258
7,bpr,Bugisnese,406,380.019380,405.0,406,258
8,bsq,Basque (Euskara),406,380.019380,405.0,406,258
9,bzc,Balinese,406,380.019380,405.0,406,258


In [51]:
# Count how many texts each font can fully render.

font_usage = []

for language, font_ids in universal_fonts_by_language.items():

    for font_id in font_ids:

        font_usage.append({
            "font_id": font_id,
            "language": language
        })


font_usage = pd.DataFrame(font_usage)


font_usage_summary = (
    font_usage
    .groupby("font_id")
    .agg(
        texts_fully_rendered=("language", "nunique")
    )
    .reset_index()
    .merge(
        font_table[
            ["font_id", "font_name"]
        ].drop_duplicates("font_id"),
        on="font_id",
        how="left"
    )
    .sort_values(
        "texts_fully_rendered",
        ascending=False
    )
    .reset_index(drop=True)
)


print("TOP 10 MOST WIDELY SUPPORTED FONTS")

display(
    font_usage_summary.head(10)
)

TOP 10 MOST WIDELY SUPPORTED FONTS


,font_id,texts_fully_rendered,font_name
0,86,257,Noto Sans Mono SemiCondensed ExtraBold
1,2288,257,Noto Sans Mono Condensed Black
2,2211,257,Noto Sans Mono ExtraCondensed Thin
3,2213,257,Noto Sans Mono ExtraCondensed SemiBold
4,1671,257,Noto Sans Mono ExtraCondensed Black
5,1696,257,Noto Sans Mono Condensed ExtraLight
6,1891,257,Noto Sans Mono SemiCondensed Light
7,1444,257,Noto Sans Mono Condensed Medium
8,1344,257,Noto Sans Mono Condensed Light
9,1345,257,Noto Sans Mono Light


In [52]:
# Determine glyph-complexity coverage for every text
# across every REAL FONT currently present in the glyph database.
#
# Morse and Braille representation records have no font_id
# and are deliberately excluded here.

available_fonts = (
    glyph_complexity_database[
        glyph_complexity_database[
            "font_id"
        ].notna()
    ][
        [
            "font_id",
            "font_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("font_id")
    .reset_index(drop=True)
)


coverage_records = []


for language, inventory in character_inventories.items():

    non_whitespace = inventory[
        ~inventory["is_whitespace"]
    ]


    required_codepoints = set(
        non_whitespace["codepoint"]
    )


    for _, font_row in available_fonts.iterrows():

        font_id = font_row["font_id"]
        font_name = font_row["font_name"]


        font_records = (
            glyph_complexity_database[
                glyph_complexity_database[
                    "font_id"
                ] == font_id
            ]
        )


        available_codepoints = set(
            font_records.loc[
                font_records[
                    "complexity"
                ].notna(),
                "codepoint"
            ]
        )


        missing_codepoints = (
            required_codepoints
            - available_codepoints
        )


        renderable = (
            len(required_codepoints)
            - len(missing_codepoints)
        )


        coverage_percent = (
            100 * renderable
            / len(required_codepoints)
            if required_codepoints
            else 100.0
        )


        coverage_records.append({

            "language":
                language,

            "font_id":
                font_id,

            "font_name":
                font_name,

            "required_characters":
                len(required_codepoints),

            "renderable":
                renderable,

            "missing":
                len(missing_codepoints),

            "coverage_percent":
                coverage_percent,

            "full_coverage":
                len(missing_codepoints) == 0

        })


coverage_table = pd.DataFrame(
    coverage_records
)


print()
print("FONT COVERAGE BY TEXT")
print()

display(
    coverage_table
)


FONT COVERAGE BY TEXT



,language,font_id,font_name,required_characters,renderable,missing,coverage_percent,full_coverage
0,abk,827,Noto Sans Thai Regular,58,1,57,1.724138,False
1,abk,1050,Noto Sans Hebrew Regular,58,1,57,1.724138,False
2,abk,1344,Noto Sans Mono Condensed Light,58,58,0,100.000000,True
3,abk,1667,Noto Sans Arabic Regular,58,13,45,22.413793,False
4,abk,2142,Noto Sans CJK SC,58,58,0,100.000000,True
...,...,...,...,...,...,...,...,...
2067,zuu,1667,Noto Sans Arabic Regular,60,13,47,21.666667,False
2068,zuu,2142,Noto Sans CJK SC,60,60,0,100.000000,True
2069,zuu,2211,Noto Sans Mono ExtraCondensed Thin,60,60,0,100.000000,True
2070,zuu,2213,Noto Sans Mono ExtraCondensed SemiBold,60,60,0,100.000000,True


In [53]:
# Remove texts that cannot be fully analyzed by ANY font
# in the glyph-complexity database.

complete_font_by_text = (
    coverage_table
    .groupby("language")["full_coverage"]
    .any()
)

zero_coverage_languages = (
    complete_font_by_text[
        ~complete_font_by_text
    ]
    .index
    .tolist()
)


if zero_coverage_languages:

    print(
        f"Removing {len(zero_coverage_languages):,} "
        f"text(s) with no fully-rendering font:"
    )

    for language in zero_coverage_languages:

        print(
            f"  {language} — "
            f"{language_names.get(language, 'UNKNOWN')}"
        )


    # Remove from loaded texts

    for language in zero_coverage_languages:

        texts.pop(
            language,
            None
        )


    # Remove character inventories

    for language in zero_coverage_languages:

        character_inventories.pop(
            language,
            None
        )


    # Remove from text file list

    text_files = [
        filepath
        for filepath in text_files
        if os.path.splitext(
            os.path.basename(filepath)
        )[0]
        not in zero_coverage_languages
    ]


    # Remove from coverage table

    coverage_table = coverage_table[
        ~coverage_table["language"].isin(
            zero_coverage_languages
        )
    ].reset_index(drop=True)


else:

    print(
        "No texts need to be removed."
    )


print()
print(
    f"Texts remaining for downstream analysis: "
    f"{len(texts):,}"
)

# add to excluded list

excluded_files.extend(
    [
        f"{language}.txt"
        for language in zero_coverage_languages
        if f"{language}.txt" not in excluded_files
    ]
)

Removing 23 text(s) with no fully-rendering font:
  aja — Adja
  ako — Wama
  ame — Amuesha-Yanesha
  bba — Batonu (Bariba)
  csa — Chinanteco
  den — Dendi
  dga — Dagaare
  dinka — Dinka
  ewe — Ewe/Eve
  foa — Fon
  gac1 — Dangme
  gac2 — Ga
  hna — Guen (Mina)
  igr — Igbo
  kea — Crioulo (Cabo Verde)
  kng — Fiote (Angola)
  lat — Latvian
  maz — Jñatrjo (Mazahua)
  nso — Lamnso' (Lám nso')
  ses — Seereer
  tbz — Ditammari
  tso — Changane (Mozambique)
  tzm — Tamazight (Beraber)

Texts remaining for downstream analysis: 236


In [54]:
# # Font lookup
# # Edit this to search for a font name if necessary

# font_search = "Noto Sans Thai Regular"

# font_lookup = font_table[
#     font_table["font_name"]
#     .str.contains(
#         font_search,
#         case=False,
#         na=False
#     )
# ].copy()

# display(
#     font_lookup[
#         [
#             "font_id",
#             "font_name",
#             "family_name",
#             "font_path",
#             "font_fileindex"
#         ]
#     ]
# )


In [55]:
# Create fast glyph-complexity lookup for fonts
#
# Key = font_id × codepoint
#
# Morse and Braille records have no font_id and are excluded

glyph_lookup = (
    glyph_complexity_database[
        glyph_complexity_database[
            "font_id"
        ].notna()
    ][
        [
            "font_id",
            "font_name",
            "codepoint",
            "complexity"
        ]
    ]
    .drop_duplicates(
        subset=[
            "font_id",
            "codepoint"
        ]
    )
    .set_index(
        [
            "font_id",
            "codepoint"
        ]
    )
)


print(
    f"Glyph lookup entries: "
    f"{len(glyph_lookup):,}"
)

Glyph lookup entries: 58,228


In [56]:
# ============================================================
# CREATE MORSE + UEB COMPLEXITY LOOKUPS
# ============================================================
#
# Morse and Unified English Braille were measured in the
# glyph-complexity notebook and stored in the SAME
# glyph_complexity_database.pkl.
#
# Unlike ordinary glyphs, these representations do not have
# font IDs. They are identified by:
#
#     representation_system + source codepoint
#
# These lookups allow the English text to be analyzed using
# the pre-calculated Morse or Braille complexity of each
# source character.
#
# This analysis is OPTIONAL and runs only when English
# ("eng") is present in character_inventories.
# ============================================================


run_english_representation_analysis = (
    "eng" in character_inventories
)


# ============================================================
# ENGLISH PRESENT
# ============================================================

if run_english_representation_analysis:

    print()
    print(
        "English detected — running Morse + UEB analysis."
    )


    # --------------------------------------------------------
    # Representation systems to analyze
    # --------------------------------------------------------

    representation_systems = [
        "International Morse Code",
        "Unified English Braille"
    ]


    # --------------------------------------------------------
    # Verify that the glyph database contains the required
    # representation columns.
    # --------------------------------------------------------

    required_representation_columns = {
        "representation_system",
        "representation_standard",
        "representation",
        "codepoint",
        "character",
        "complexity"
    }


    missing_representation_columns = (
        required_representation_columns
        - set(glyph_complexity_database.columns)
    )


    if missing_representation_columns:

        raise ValueError(
            "Glyph complexity database does not contain the "
            "required Morse/UEB columns:\n"
            f"{sorted(missing_representation_columns)}\n\n"
            "Run the updated glyph-complexity notebook first."
        )


    # --------------------------------------------------------
    # Extract Morse + UEB records
    # --------------------------------------------------------

    representation_complexity_database = (
        glyph_complexity_database[
            glyph_complexity_database[
                "representation_system"
            ].isin(
                representation_systems
            )
        ]
        .copy()
    )


    if representation_complexity_database.empty:

        raise ValueError(
            "No Morse or UEB records were found in "
            "glyph_complexity_database.pkl.\n"
            "Run the updated glyph-complexity notebook first."
        )


    # --------------------------------------------------------
    # Build one lookup per representation system.
    #
    # Key:
    #     source Unicode codepoint
    #
    # Value:
    #     pre-calculated representation complexity
    # --------------------------------------------------------

    representation_lookups = {}


    for system in representation_systems:

        system_data = (
            representation_complexity_database[
                representation_complexity_database[
                    "representation_system"
                ] == system
            ]
            .copy()
        )


        # Require records for every requested system.

        if system_data.empty:

            raise ValueError(
                f"No records were found for {system}."
            )


        # There should be only one representation per
        # source character within each representation system.

        duplicate_codepoints = (
            system_data[
                system_data[
                    "codepoint"
                ].duplicated(
                    keep=False
                )
            ]
        )


        if not duplicate_codepoints.empty:

            raise ValueError(
                f"Duplicate source characters found for {system}."
            )


        representation_lookups[system] = (
            system_data[
                [
                    "codepoint",
                    "character",
                    "representation",
                    "complexity"
                ]
            ]
            .set_index(
                "codepoint"
            )
        )


    # --------------------------------------------------------
    # Report available representations
    # --------------------------------------------------------

    print()
    print("=" * 80)
    print("MORSE + UEB COMPLEXITY LOOKUPS")
    print("=" * 80)


    for system in representation_systems:

        lookup = representation_lookups[
            system
        ]

        measurable = int(
            lookup["complexity"]
            .notna()
            .sum()
        )

        print()
        print(system)

        print(
            f"  Source characters represented: "
            f"{len(lookup):,}"
        )

        print(
            f"  Characters with complexity scores: "
            f"{measurable:,}"
        )


# Reports representation coverage of the English source text.
#
# Complete coverage is NOT required.
# Unsupported characters are identified here and are later
# excluded from both sides of the corresponding matched
# English-font vs representation comparison.

    representation_coverage_records = []


    english_inventory = (
        character_inventories[
            "eng"
        ]
        .copy()
    )


    english_nonwhitespace = (
        english_inventory[
            ~english_inventory[
                "is_whitespace"
            ]
        ]
        .copy()
    )


    for system in representation_systems:

        lookup = representation_lookups[
            system
        ]


        missing_characters = []

        represented_characters = 0


        for _, row in english_nonwhitespace.iterrows():

            source_character = row[
                "character"
            ]


            # International Morse does not distinguish
            # uppercase and lowercase Latin letters.

            if system == "International Morse Code":

                lookup_character = (
                    source_character.upper()
                )

            else:

                lookup_character = (
                    source_character
                )


            # A one-character lookup is required.
            #
            # Characters whose uppercase transformation
            # produces multiple characters (for example ß)
            # are NOT silently transliterated.

            if len(lookup_character) != 1:

                missing_characters.append(
                    source_character
                )

                continue


            lookup_codepoint = ord(
                lookup_character
            )


            if (
                lookup_codepoint
                not in lookup.index
            ):

                missing_characters.append(
                    source_character
                )

                continue


            complexity = lookup.loc[
                lookup_codepoint,
                "complexity"
            ]


            if pd.isna(complexity):

                missing_characters.append(
                    source_character
                )

                continue


            represented_characters += int(
                row["count"]
            )


        total_nonwhitespace = int(
            english_nonwhitespace[
                "count"
            ].sum()
        )


        missing_characters = sorted(
            set(
                missing_characters
            )
        )


        full_coverage = (
            len(missing_characters) == 0
        )


        representation_coverage_records.append(
            {
                "language":
                    "eng",

                "language_name":
                    language_names.get(
                        "eng",
                        "English"
                    ),

                "representation_system":
                    system,

                "nonwhitespace_characters":
                    total_nonwhitespace,

                "represented_characters":
                    represented_characters,

                "full_coverage":
                    full_coverage,

                "missing_characters":
                    "".join(
                        missing_characters
                    )
            }
        )


    representation_coverage_table = pd.DataFrame(
        representation_coverage_records
    )


    print()
    print("ENGLISH REPRESENTATION COVERAGE")
    print()

    display(
        representation_coverage_table
    )


# ============================================================
# ENGLISH NOT PRESENT
# ============================================================

else:

    print()
    print(
        "English not present — skipping Morse + UEB analysis."
    )


    # Define empty objects so downstream cells can safely
    # test them without encountering undefined variables.

    representation_systems = []

    representation_complexity_database = pd.DataFrame()

    representation_lookups = {}

    representation_coverage_records = []

    representation_coverage_table = pd.DataFrame()


English detected — running Morse + UEB analysis.

MORSE + UEB COMPLEXITY LOOKUPS

International Morse Code
  Source characters represented: 50
  Characters with complexity scores: 49

Unified English Braille
  Source characters represented: 86
  Characters with complexity scores: 85

ENGLISH REPRESENTATION COVERAGE



,language,language_name,representation_system,nonwhitespace_characters,represented_characters,full_coverage,missing_characters
0,eng,English,International Morse Code,8952,8948,False,;
1,eng,English,Unified English Braille,8952,8952,True,


In [57]:
# Select fonts for downstream analysis.
#
# A font only needs to have glyph-complexity measurements in the
# glyph database. Individual texts may or may not be fully covered
# by each selected font.

analysis_fonts = [
    2142,
    1344,
    2211,
    2213,
    2288,
    1667,
    1050,
    827
]


available_font_ids = set(
    glyph_complexity_database["font_id"]
    .dropna()
)


valid_analysis_fonts = [
    font_id
    for font_id in analysis_fonts
    if font_id in available_font_ids
]


invalid_analysis_fonts = [
    font_id
    for font_id in analysis_fonts
    if font_id not in available_font_ids
]


print(
    f"Requested analysis fonts: "
    f"{len(analysis_fonts):,}"
)

print(
    f"Available in glyph database: "
    f"{len(valid_analysis_fonts):,}"
)

print(
    f"Not found in glyph database: "
    f"{len(invalid_analysis_fonts):,}"
)


if invalid_analysis_fonts:

    print()
    print("INVALID FONT IDs:")
    print(invalid_analysis_fonts)


analysis_fonts = valid_analysis_fonts

Requested analysis fonts: 8
Available in glyph database: 8
Not found in glyph database: 0


In [58]:
# Display selected analysis fonts.

selected_font_info = (
    font_table[
        font_table["font_id"].isin(
            analysis_fonts
        )
    ]
    [
        [
            "font_id",
            "font_name",
            "family_name",
            "font_path"
        ]
    ]
    .drop_duplicates(
        subset=["font_id"]
    )
    .sort_values(
        ["family_name", "font_name", "font_id"]
    )
    .reset_index(drop=True)
)


display(
    selected_font_info
)

,font_id,font_name,family_name,font_path
0,1667,Noto Sans Arabic Regular,Noto Sans Arabic,/usr/share/fonts/truetype/noto/NotoSansArabic-...
1,2142,Noto Sans CJK SC,Noto Sans CJK SC,/usr/share/fonts/opentype/noto/NotoSansCJK-Reg...
2,1050,Noto Sans Hebrew Regular,Noto Sans Hebrew,/usr/share/fonts/truetype/noto/NotoSansHebrew-...
3,2288,Noto Sans Mono Condensed Black,Noto Sans Mono Condensed Black,/usr/share/fonts/truetype/noto/NotoSansMono-Co...
4,1344,Noto Sans Mono Condensed Light,Noto Sans Mono Condensed Light,/usr/share/fonts/truetype/noto/NotoSansMono-Co...
5,2213,Noto Sans Mono ExtraCondensed SemiBold,Noto Sans Mono ExtraCondensed SemiBold,/usr/share/fonts/truetype/noto/NotoSansMono-Ex...
6,2211,Noto Sans Mono ExtraCondensed Thin,Noto Sans Mono ExtraCondensed Thin,/usr/share/fonts/truetype/noto/NotoSansMono-Ex...
7,827,Noto Sans Thai Regular,Noto Sans Thai,/usr/share/fonts/truetype/noto/NotoSansThai-Re...


In [59]:
# Coverage of each selected font for each text.
#
# A text/font pair is usable only when every non-whitespace character
# in that text has a non-null glyph-complexity measurement.

selected_coverage_table = (
    coverage_table[
        coverage_table["font_id"].isin(
            analysis_fonts
        )
    ]
    .copy()
    .reset_index(drop=True)
)


display(
    selected_coverage_table
)

,language,font_id,font_name,required_characters,renderable,missing,coverage_percent,full_coverage
0,abk,827,Noto Sans Thai Regular,58,1,57,1.724138,False
1,abk,1050,Noto Sans Hebrew Regular,58,1,57,1.724138,False
2,abk,1344,Noto Sans Mono Condensed Light,58,58,0,100.000000,True
3,abk,1667,Noto Sans Arabic Regular,58,13,45,22.413793,False
4,abk,2142,Noto Sans CJK SC,58,58,0,100.000000,True
...,...,...,...,...,...,...,...,...
1883,zuu,1667,Noto Sans Arabic Regular,60,13,47,21.666667,False
1884,zuu,2142,Noto Sans CJK SC,60,60,0,100.000000,True
1885,zuu,2211,Noto Sans Mono ExtraCondensed Thin,60,60,0,100.000000,True
1886,zuu,2213,Noto Sans Mono ExtraCondensed SemiBold,60,60,0,100.000000,True


In [60]:
# Keep only text × font combinations with complete glyph-complexity coverage.
#
# Texts remain available for analysis with other fonts that do have
# complete coverage.

selected_coverage_table = (
    selected_coverage_table[
        selected_coverage_table["full_coverage"]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Usable text × font combinations: "
    f"{len(selected_coverage_table):,}"
)

print(
    f"Texts represented: "
    f"{selected_coverage_table['language'].nunique():,}"
)

print(
    f"Fonts represented: "
    f"{selected_coverage_table['font_id'].nunique():,}"
)


display(
    selected_coverage_table
)

Usable text × font combinations: 1,123
Texts represented: 236
Fonts represented: 5


,language,font_id,font_name,required_characters,renderable,missing,coverage_percent,full_coverage
0,abk,1344,Noto Sans Mono Condensed Light,58,58,0,100.0,True
1,abk,2142,Noto Sans CJK SC,58,58,0,100.0,True
2,abk,2211,Noto Sans Mono ExtraCondensed Thin,58,58,0,100.0,True
3,abk,2213,Noto Sans Mono ExtraCondensed SemiBold,58,58,0,100.0,True
4,abk,2288,Noto Sans Mono Condensed Black,58,58,0,100.0,True
...,...,...,...,...,...,...,...,...
1118,zuu,1344,Noto Sans Mono Condensed Light,60,60,0,100.0,True
1119,zuu,2142,Noto Sans CJK SC,60,60,0,100.0,True
1120,zuu,2211,Noto Sans Mono ExtraCondensed Thin,60,60,0,100.0,True
1121,zuu,2213,Noto Sans Mono ExtraCondensed SemiBold,60,60,0,100.0,True


In [61]:
# CALCULATE FONT-BASED TEXT + CHARACTER COMPLEXITY
#
# Produces two datasets:
#
# 1. text_complexity
#       One row per text × font.
#
# 2. character_complexity
#       One row per source character × text × font.
#
# For each character:
#
#     weighted_complexity =
#         character count × glyph complexity
#
# Text complexity:
#
#     total_complexity =
#         sum(weighted_complexity)
#
#     normalized_complexity =
#         total_complexity / non-whitespace source characters
#
# Whitespace is excluded from complexity calculations.



text_complexity_records = []

character_complexity_records = []


for _, coverage_row in selected_coverage_table.iterrows():

    language = coverage_row["language"]

    font_id = coverage_row["font_id"]

    font_name = coverage_row["font_name"]


    inventory = character_inventories[
        language
    ]


    total_characters = int(
        inventory["count"].sum()
    )


    whitespace_characters = int(
        inventory.loc[
            inventory["is_whitespace"],
            "count"
        ].sum()
    )


    nonwhitespace_characters = (
        total_characters
        - whitespace_characters
    )


    # Character-type counts

    letters = int(
        inventory.loc[
            inventory["category"].str.startswith("L"),
            "count"
        ].sum()
    )


    numbers = int(
        inventory.loc[
            inventory["category"].str.startswith("N"),
            "count"
        ].sum()
    )


    punctuation = int(
        inventory.loc[
            inventory["category"].str.startswith("P"),
            "count"
        ].sum()
    )


    symbols = int(
        inventory.loc[
            inventory["category"].str.startswith("S"),
            "count"
        ].sum()
    )


    # Calculate character-level complexity

    nonwhitespace_inventory = (
        inventory[
            ~inventory["is_whitespace"]
        ]
        .copy()
    )


    total_complexity = 0.0


    for _, char_row in nonwhitespace_inventory.iterrows():

        key = (
            font_id,
            char_row["codepoint"]
        )


        complexity = glyph_lookup.loc[
            key,
            "complexity"
        ]


        count = int(
            char_row["count"]
        )


        weighted_complexity = (
            count
            * float(complexity)
        )


        total_complexity += (
            weighted_complexity
        )


        # Preserve the character-level result.

        character_complexity_records.append(
            {
                "language":
                    language,

                "language_name":
                    language_names[language],

                "font_id":
                    font_id,

                "font_name":
                    font_name,

                "character":
                    char_row["character"],

                "codepoint":
                    int(char_row["codepoint"]),

                "unicode":
                    char_row["unicode"],

                "unicode_name":
                    char_row["unicode_name"],

                "category":
                    char_row["category"],

                "count":
                    count,

                "complexity":
                    float(complexity),

                "weighted_complexity":
                    weighted_complexity
            }
        )


    # Normalize text complexity

    normalized_complexity = (
        total_complexity
        / nonwhitespace_characters
        if nonwhitespace_characters > 0
        else np.nan
    )


    # Preserve text-level result

    text_complexity_records.append(
        {
            "language":
                language,

            "language_name":
                language_names[language],

            "font_id":
                font_id,

            "font_name":
                font_name,

            "total_characters":
                total_characters,

            "nonwhitespace_characters":
                nonwhitespace_characters,

            "whitespace_characters":
                whitespace_characters,

            "letters":
                letters,

            "numbers":
                numbers,

            "punctuation":
                punctuation,

            "symbols":
                symbols,

            "unique_characters":
                len(inventory),

            "unique_nonwhitespace_characters":
                len(nonwhitespace_inventory),

            "total_complexity":
                total_complexity,

            "normalized_complexity":
                normalized_complexity
        }
    )


# CREATE DATAFRAMES

# total_complexity = raw cumulative complexity
text_complexity = pd.DataFrame(
    text_complexity_records
)


character_complexity = pd.DataFrame(
    character_complexity_records
)


print()
print(
    f"Text × font measurements: "
    f"{len(text_complexity):,}"
)

print(
    f"Character × text × font measurements: "
    f"{len(character_complexity):,}"
)



Text × font measurements: 1,123
Character × text × font measurements: 67,895


In [62]:
# ============================================================
# SANITY CHECK TEXT COMPLEXITY MEASUREMENTS
# ============================================================
#
# Verify directly that every non-whitespace source character
# belongs to a Unicode category included in the analysis:
#
#     L = Letter
#     N = Number
#     P = Punctuation
#     S = Symbol
#
# This is more informative than checking whether
#
#     letters + numbers + punctuation + symbols
#
# equals the total non-whitespace count, because unexpected
# characters are identified explicitly.


allowed_category_prefixes = (
    "L",
    "N",
    "P",
    "S"
)


unexpected_character_records = []


for language, inventory in character_inventories.items():

    unexpected = inventory[
        (~inventory["is_whitespace"])
        &
        (~inventory["category"].str.startswith(
            allowed_category_prefixes
        ))
    ].copy()


    if not unexpected.empty:

        for _, row in unexpected.iterrows():

            unexpected_character_records.append(
                {
                    "language":
                        language,

                    "language_name":
                        row["language_name"],

                    "character":
                        row["character"],

                    "unicode":
                        row["unicode"],

                    "unicode_name":
                        row["unicode_name"],

                    "category":
                        row["category"],

                    "count":
                        row["count"]
                }
            )


unexpected_characters = pd.DataFrame(
    unexpected_character_records
)


if unexpected_characters.empty:

    print(
        "Character-category sanity check passed."
    )

    print(
        "Every non-whitespace character belongs to "
        "Unicode category L, N, P, or S."
    )

else:

    print(
        "CHARACTER-CATEGORY SANITY CHECK FAILED"
    )

    print()

    print(
        "The following non-whitespace characters fall outside "
        "Unicode categories L, N, P, and S:"
    )

    print()

    display(
        unexpected_characters
    )

    raise ValueError(
        "Unexpected non-whitespace Unicode categories found. "
        "Inspect unexpected_characters before continuing."
    )

Character-category sanity check passed.
Every non-whitespace character belongs to Unicode category L, N, P, or S.


In [63]:
# Primary text-complexity table.

text_complexity_table = (
    text_complexity[
        [
            "language",
            "language_name",
            "font_id",
            "font_name",
            "normalized_complexity",
            "total_complexity",
            "total_characters",
            "nonwhitespace_characters",
            "whitespace_characters",
            "letters",
            "numbers",
            "punctuation",
            "symbols",
            "unique_characters",
            "unique_nonwhitespace_characters"
        ]
    ]
    .sort_values(
        [
            "language",
            "font_id"
        ]
    )
    .reset_index(drop=True)
)


display(
    text_complexity_table
)

,language,language_name,font_id,font_name,normalized_complexity,total_complexity,total_characters,nonwhitespace_characters,whitespace_characters,letters,numbers,punctuation,symbols,unique_characters,unique_nonwhitespace_characters
0,abk,Abkhaz,1344,Noto Sans Mono Condensed Light,0.196106,1974.200426,11446,10067,1379,9124,661,254,28,61,58
1,abk,Abkhaz,2142,Noto Sans CJK SC,0.197299,1986.204458,11446,10067,1379,9124,661,254,28,61,58
2,abk,Abkhaz,2211,Noto Sans Mono ExtraCondensed Thin,0.206150,2075.315466,11446,10067,1379,9124,661,254,28,61,58
3,abk,Abkhaz,2213,Noto Sans Mono ExtraCondensed SemiBold,0.188174,1894.348110,11446,10067,1379,9124,661,254,28,61,58
4,abk,Abkhaz,2288,Noto Sans Mono Condensed Black,0.184167,1854.010377,11446,10067,1379,9124,661,254,28,61,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,zuu,Zulu,1344,Noto Sans Mono Condensed Light,0.189168,1750.941383,10381,9256,1125,8980,83,193,0,63,60
1119,zuu,Zulu,2142,Noto Sans CJK SC,0.190106,1759.622517,10381,9256,1125,8980,83,193,0,63,60
1120,zuu,Zulu,2211,Noto Sans Mono ExtraCondensed Thin,0.197316,1826.361142,10381,9256,1125,8980,83,193,0,63,60
1121,zuu,Zulu,2213,Noto Sans Mono ExtraCondensed SemiBold,0.183410,1697.641963,10381,9256,1125,8980,83,193,0,63,60


In [66]:
# ============================================================
# ENGLISH REPRESENTATION COMPARISON — SETUP
# ============================================================
#
# Morse and UEB are analyzed separately.
#
# IMPORTANT MATCHING RULE:
#
# If a source English character cannot be represented by a
# representation system, that character is excluded from BOTH:
#
#   1. the representation-system calculation
#   2. the English-font comparison calculation
#
# This produces matched source-character populations for each
# pairwise comparison.
#
# Therefore:
#
#   English vs Morse
#       uses only source characters supported by Morse.
#
#   English vs UEB
#       uses only source characters supported by UEB.
#
# The Morse and UEB comparison populations may differ.


representation_text_complexity_records = []

representation_character_complexity_records = []

representation_exclusion_records = []


if run_english_representation_analysis:

    english_inventory = (
        character_inventories[
            "eng"
        ]
        .copy()
    )


    english_nonwhitespace = (
        english_inventory[
            ~english_inventory[
                "is_whitespace"
            ]
        ]
        .copy()
    )


    total_characters = int(
        english_inventory[
            "count"
        ].sum()
    )


    whitespace_characters = int(
        english_inventory.loc[
            english_inventory[
                "is_whitespace"
            ],
            "count"
        ].sum()
    )


    total_nonwhitespace_characters = int(
        english_nonwhitespace[
            "count"
        ].sum()
    )


    print()
    print("=" * 70)
    print("ENGLISH REPRESENTATION COMPARISON")
    print("=" * 70)

    print()
    print(
        f"Total source characters: "
        f"{total_characters:,}"
    )

    print(
        f"Non-whitespace source characters: "
        f"{total_nonwhitespace_characters:,}"
    )

else:

    english_inventory = pd.DataFrame()

    english_nonwhitespace = pd.DataFrame()

    total_characters = 0

    whitespace_characters = 0

    total_nonwhitespace_characters = 0


ENGLISH REPRESENTATION COMPARISON

Total source characters: 10,746
Non-whitespace source characters: 8,952


In [67]:
# ============================================================
# INTERNATIONAL MORSE CODE
# ============================================================
#
# Morse is case-insensitive.
#
# Unsupported source characters are explicitly identified and
# excluded from the Morse comparison population.
#
# They must ALSO be excluded later from the English-font side
# of the English-vs-Morse comparison.


morse_character_complexity_records = []

morse_exclusion_records = []


if run_english_representation_analysis:

    system = "International Morse Code"

    lookup = representation_lookups[
        system
    ]


    # --------------------------------------------------------
    # Determine representation standard
    # --------------------------------------------------------

    standard_values = (
        representation_complexity_database.loc[
            representation_complexity_database[
                "representation_system"
            ] == system,
            "representation_standard"
        ]
        .dropna()
        .unique()
    )


    representation_standard = (
        standard_values[0]
        if len(standard_values) == 1
        else ""
    )


    # --------------------------------------------------------
    # Determine supported and unsupported source characters
    # --------------------------------------------------------

    supported_rows = []

    unsupported_rows = []


    for _, char_row in english_nonwhitespace.iterrows():

        source_character = (
            char_row["character"]
        )


        # Morse is case-insensitive.

        lookup_character = (
            source_character.upper()
        )


        supported = True

        exclusion_reason = None


        # Do not silently transliterate characters whose
        # uppercase transformation produces multiple characters.

        if len(lookup_character) != 1:

            supported = False

            exclusion_reason = (
                "uppercase mapping is not one character"
            )


        else:

            lookup_codepoint = ord(
                lookup_character
            )


            if lookup_codepoint not in lookup.index:

                supported = False

                exclusion_reason = (
                    "no standardized Morse representation"
                )


            elif pd.isna(
                lookup.loc[
                    lookup_codepoint,
                    "complexity"
                ]
            ):

                supported = False

                exclusion_reason = (
                    "Morse representation has no complexity score"
                )


        if supported:

            supported_rows.append(
                char_row
            )

        else:

            exclusion_record = {
                "representation_system":
                    system,

                "character":
                    source_character,

                "codepoint":
                    int(char_row["codepoint"]),

                "unicode":
                    char_row["unicode"],

                "unicode_name":
                    char_row["unicode_name"],

                "category":
                    char_row["category"],

                "count":
                    int(char_row["count"]),

                "reason":
                    exclusion_reason
            }

            unsupported_rows.append(
                char_row
            )

            morse_exclusion_records.append(
                exclusion_record
            )

            representation_exclusion_records.append(
                exclusion_record
            )


    morse_supported_inventory = (
        pd.DataFrame(
            supported_rows
        )
        .reset_index(drop=True)
    )


    morse_excluded_inventory = (
        pd.DataFrame(
            unsupported_rows
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Report exclusions
    # --------------------------------------------------------

    morse_supported_characters = int(
        morse_supported_inventory[
            "count"
        ].sum()
    )


    morse_excluded_characters = (
        total_nonwhitespace_characters
        - morse_supported_characters
    )


    morse_coverage_percent = (
        100
        * morse_supported_characters
        / total_nonwhitespace_characters
        if total_nonwhitespace_characters > 0
        else np.nan
    )


    print()
    print("=" * 70)
    print("INTERNATIONAL MORSE CODE COVERAGE")
    print("=" * 70)

    print()

    print(
        f"Included source-character occurrences: "
        f"{morse_supported_characters:,}"
    )

    print(
        f"Excluded source-character occurrences: "
        f"{morse_excluded_characters:,}"
    )

    print(
        f"Coverage: "
        f"{morse_coverage_percent:.4f}%"
    )

    print()


    if morse_exclusion_records:

        print(
            "Characters excluded from BOTH Morse and its "
            "English-font comparator:"
        )

        display(
            pd.DataFrame(
                morse_exclusion_records
            )
        )

    else:

        print(
            "No source characters require exclusion."
        )


    # --------------------------------------------------------
    # Calculate Morse complexity
    # --------------------------------------------------------

    total_complexity = 0.0


    for _, char_row in morse_supported_inventory.iterrows():

        source_character = (
            char_row["character"]
        )

        lookup_character = (
            source_character.upper()
        )

        lookup_codepoint = ord(
            lookup_character
        )


        complexity = float(
            lookup.loc[
                lookup_codepoint,
                "complexity"
            ]
        )


        representation = lookup.loc[
            lookup_codepoint,
            "representation"
        ]


        count = int(
            char_row["count"]
        )


        weighted_complexity = (
            count
            * complexity
        )


        total_complexity += (
            weighted_complexity
        )


        record = {
            "language":
                "eng",

            "language_name":
                language_names.get(
                    "eng",
                    "English"
                ),

            "representation_system":
                system,

            "representation_standard":
                representation_standard,

            "character":
                source_character,

            "codepoint":
                int(char_row["codepoint"]),

            "unicode":
                char_row["unicode"],

            "unicode_name":
                char_row["unicode_name"],

            "category":
                char_row["category"],

            "count":
                count,

            "representation":
                representation,

            "complexity":
                complexity,

            "weighted_complexity":
                weighted_complexity
        }


        morse_character_complexity_records.append(
            record
        )

        representation_character_complexity_records.append(
            record
        )


    morse_normalized_complexity = (
        total_complexity
        / morse_supported_characters
        if morse_supported_characters > 0
        else np.nan
    )


    representation_text_complexity_records.append(
        {
            "language":
                "eng",

            "language_name":
                language_names.get(
                    "eng",
                    "English"
                ),

            "representation_system":
                system,

            "representation_standard":
                representation_standard,

            "total_complexity":
                total_complexity,

            "normalized_complexity":
                morse_normalized_complexity,

            "source_nonwhitespace_characters":
                total_nonwhitespace_characters,

            "included_characters":
                morse_supported_characters,

            "excluded_characters":
                morse_excluded_characters,

            "coverage_percent":
                morse_coverage_percent,

            "unique_included_characters":
                len(
                    morse_supported_inventory
                ),

            "unique_excluded_characters":
                len(
                    morse_excluded_inventory
                )
        }
    )


    print()
    print(
        f"Morse normalized complexity: "
        f"{morse_normalized_complexity:.6f}"
    )


INTERNATIONAL MORSE CODE COVERAGE

Included source-character occurrences: 8,948
Excluded source-character occurrences: 4
Coverage: 99.9553%

Characters excluded from BOTH Morse and its English-font comparator:


,representation_system,character,codepoint,unicode,unicode_name,category,count,reason
0,International Morse Code,;,59,U+003B,SEMICOLON,Po,4,no standardized Morse representation



Morse normalized complexity: 0.151779


In [68]:
# ============================================================
# UNIFIED ENGLISH BRAILLE
# ============================================================
#
# UEB preserves source-character case.
#
# Unsupported source characters are explicitly identified and
# excluded from the UEB comparison population.
#
# They must ALSO be excluded later from the English-font side
# of the English-vs-UEB comparison.


ueb_character_complexity_records = []

ueb_exclusion_records = []


if run_english_representation_analysis:

    system = "Unified English Braille"

    lookup = representation_lookups[
        system
    ]


    # --------------------------------------------------------
    # Determine representation standard
    # --------------------------------------------------------

    standard_values = (
        representation_complexity_database.loc[
            representation_complexity_database[
                "representation_system"
            ] == system,
            "representation_standard"
        ]
        .dropna()
        .unique()
    )


    representation_standard = (
        standard_values[0]
        if len(standard_values) == 1
        else ""
    )


    # --------------------------------------------------------
    # Determine supported and unsupported source characters
    # --------------------------------------------------------

    supported_rows = []

    unsupported_rows = []


    for _, char_row in english_nonwhitespace.iterrows():

        source_character = (
            char_row["character"]
        )


        # UEB preserves source-character case.

        lookup_character = (
            source_character
        )


        lookup_codepoint = ord(
            lookup_character
        )


        supported = True

        exclusion_reason = None


        if lookup_codepoint not in lookup.index:

            supported = False

            exclusion_reason = (
                "no standardized UEB representation"
            )


        elif pd.isna(
            lookup.loc[
                lookup_codepoint,
                "complexity"
            ]
        ):

            supported = False

            exclusion_reason = (
                "UEB representation has no complexity score"
            )


        if supported:

            supported_rows.append(
                char_row
            )

        else:

            exclusion_record = {
                "representation_system":
                    system,

                "character":
                    source_character,

                "codepoint":
                    int(char_row["codepoint"]),

                "unicode":
                    char_row["unicode"],

                "unicode_name":
                    char_row["unicode_name"],

                "category":
                    char_row["category"],

                "count":
                    int(char_row["count"]),

                "reason":
                    exclusion_reason
            }

            unsupported_rows.append(
                char_row
            )

            ueb_exclusion_records.append(
                exclusion_record
            )

            representation_exclusion_records.append(
                exclusion_record
            )


    ueb_supported_inventory = (
        pd.DataFrame(
            supported_rows
        )
        .reset_index(drop=True)
    )


    ueb_excluded_inventory = (
        pd.DataFrame(
            unsupported_rows
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Report exclusions
    # --------------------------------------------------------

    ueb_supported_characters = int(
        ueb_supported_inventory[
            "count"
        ].sum()
    )


    ueb_excluded_characters = (
        total_nonwhitespace_characters
        - ueb_supported_characters
    )


    ueb_coverage_percent = (
        100
        * ueb_supported_characters
        / total_nonwhitespace_characters
        if total_nonwhitespace_characters > 0
        else np.nan
    )


    print()
    print("=" * 70)
    print("UNIFIED ENGLISH BRAILLE COVERAGE")
    print("=" * 70)

    print()

    print(
        f"Included source-character occurrences: "
        f"{ueb_supported_characters:,}"
    )

    print(
        f"Excluded source-character occurrences: "
        f"{ueb_excluded_characters:,}"
    )

    print(
        f"Coverage: "
        f"{ueb_coverage_percent:.4f}%"
    )

    print()


    if ueb_exclusion_records:

        print(
            "Characters excluded from BOTH UEB and its "
            "English-font comparator:"
        )

        display(
            pd.DataFrame(
                ueb_exclusion_records
            )
        )

    else:

        print(
            "No source characters require exclusion."
        )


    # --------------------------------------------------------
    # Calculate UEB complexity
    # --------------------------------------------------------

    total_complexity = 0.0


    for _, char_row in ueb_supported_inventory.iterrows():

        source_character = (
            char_row["character"]
        )

        lookup_codepoint = int(
            char_row["codepoint"]
        )


        complexity = float(
            lookup.loc[
                lookup_codepoint,
                "complexity"
            ]
        )


        representation = lookup.loc[
            lookup_codepoint,
            "representation"
        ]


        count = int(
            char_row["count"]
        )


        weighted_complexity = (
            count
            * complexity
        )


        total_complexity += (
            weighted_complexity
        )


        record = {
            "language":
                "eng",

            "language_name":
                language_names.get(
                    "eng",
                    "English"
                ),

            "representation_system":
                system,

            "representation_standard":
                representation_standard,

            "character":
                source_character,

            "codepoint":
                int(char_row["codepoint"]),

            "unicode":
                char_row["unicode"],

            "unicode_name":
                char_row["unicode_name"],

            "category":
                char_row["category"],

            "count":
                count,

            "representation":
                representation,

            "complexity":
                complexity,

            "weighted_complexity":
                weighted_complexity
        }


        ueb_character_complexity_records.append(
            record
        )

        representation_character_complexity_records.append(
            record
        )


    ueb_normalized_complexity = (
        total_complexity
        / ueb_supported_characters
        if ueb_supported_characters > 0
        else np.nan
    )


    representation_text_complexity_records.append(
        {
            "language":
                "eng",

            "language_name":
                language_names.get(
                    "eng",
                    "English"
                ),

            "representation_system":
                system,

            "representation_standard":
                representation_standard,

            "total_complexity":
                total_complexity,

            "normalized_complexity":
                ueb_normalized_complexity,

            "source_nonwhitespace_characters":
                total_nonwhitespace_characters,

            "included_characters":
                ueb_supported_characters,

            "excluded_characters":
                ueb_excluded_characters,

            "coverage_percent":
                ueb_coverage_percent,

            "unique_included_characters":
                len(
                    ueb_supported_inventory
                ),

            "unique_excluded_characters":
                len(
                    ueb_excluded_inventory
                )
        }
    )


    print()
    print(
        f"UEB normalized complexity: "
        f"{ueb_normalized_complexity:.6f}"
    )


UNIFIED ENGLISH BRAILLE COVERAGE

Included source-character occurrences: 8,952
Excluded source-character occurrences: 0
Coverage: 100.0000%

No source characters require exclusion.

UEB normalized complexity: 0.191222


In [69]:
# ============================================================
# COMBINE REPRESENTATION RESULTS + INTEGRITY CHECKS
# ============================================================


representation_text_complexity = pd.DataFrame(
    representation_text_complexity_records
)


representation_character_complexity = pd.DataFrame(
    representation_character_complexity_records
)


representation_exclusions = pd.DataFrame(
    representation_exclusion_records
)


if run_english_representation_analysis:

    print()
    print("=" * 70)
    print("REPRESENTATION TEXT COMPLEXITY")
    print("=" * 70)
    print()

    display(
        representation_text_complexity
    )


    print()
    print("=" * 70)
    print("REPRESENTATION CHARACTER COMPLEXITY")
    print("=" * 70)
    print()

    display(
        representation_character_complexity
    )


    print()
    print("=" * 70)
    print("EXCLUDED SOURCE CHARACTERS")
    print("=" * 70)
    print()


    if representation_exclusions.empty:

        print(
            "No source characters were excluded."
        )

    else:

        display(
            representation_exclusions
            .sort_values(
                [
                    "representation_system",
                    "codepoint"
                ]
            )
            .reset_index(drop=True)
        )


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    for _, result in (
        representation_text_complexity.iterrows()
    ):

        system = result[
            "representation_system"
        ]

        included = int(
            result[
                "included_characters"
            ]
        )

        excluded = int(
            result[
                "excluded_characters"
            ]
        )


        if (
            included
            + excluded
            != total_nonwhitespace_characters
        ):

            raise ValueError(
                f"{system}: included + excluded character "
                "counts do not equal the source "
                "non-whitespace character count."
            )


        character_rows = (
            representation_character_complexity[
                representation_character_complexity[
                    "representation_system"
                ] == system
            ]
        )


        reconstructed_count = int(
            character_rows[
                "count"
            ].sum()
        )


        if reconstructed_count != included:

            raise ValueError(
                f"{system}: character-level counts do not "
                "match the included comparison population."
            )


        reconstructed_total = float(
            character_rows[
                "weighted_complexity"
            ].sum()
        )


        if not np.isclose(
            reconstructed_total,
            result[
                "total_complexity"
            ]
        ):

            raise ValueError(
                f"{system}: character-level weighted "
                "complexity does not reconstruct the "
                "text-level total."
            )


    print()
    print(
        "Representation integrity checks passed."
    )


else:

    print()
    print(
        "English not present — "
        "Morse + UEB complexity analysis skipped."
    )


REPRESENTATION TEXT COMPLEXITY



,language,language_name,representation_system,representation_standard,total_complexity,normalized_complexity,source_nonwhitespace_characters,included_characters,excluded_characters,coverage_percent,unique_included_characters,unique_excluded_characters
0,eng,English,International Morse Code,ITU-R M.1677-1,1358.122403,0.151779,8952,8948,4,99.955317,55,1
1,eng,English,Unified English Braille,UEB,1711.817488,0.191222,8952,8952,0,100.000000,56,0



REPRESENTATION CHARACTER COMPLEXITY



,language,language_name,representation_system,representation_standard,character,codepoint,unicode,unicode_name,category,count,representation,complexity,weighted_complexity
0,eng,English,International Morse Code,ITU-R M.1677-1,e,101,U+0065,LATIN SMALL LETTER E,Ll,1045,.,0.126500,132.192793
1,eng,English,International Morse Code,ITU-R M.1677-1,t,116,U+0074,LATIN SMALL LETTER T,Ll,795,-,0.091467,72.715883
2,eng,English,International Morse Code,ITU-R M.1677-1,o,111,U+006F,LATIN SMALL LETTER O,Ll,706,---,0.147405,104.068001
3,eng,English,International Morse Code,ITU-R M.1677-1,n,110,U+006E,LATIN SMALL LETTER N,Ll,698,-.,0.136263,95.111704
4,eng,English,International Morse Code,ITU-R M.1677-1,i,105,U+0069,LATIN SMALL LETTER I,Ll,696,..,0.151219,105.248191
...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,eng,English,Unified English Braille,UEB,R,82,U+0052,LATIN CAPITAL LETTER R,Lu,2,⠠⠗,0.224529,0.449059
107,eng,English,Unified English Braille,UEB,H,72,U+0048,LATIN CAPITAL LETTER H,Lu,2,⠠⠓,0.214906,0.429813
108,eng,English,Unified English Braille,UEB,C,67,U+0043,LATIN CAPITAL LETTER C,Lu,1,⠠⠉,0.203696,0.203696
109,eng,English,Unified English Braille,UEB,G,71,U+0047,LATIN CAPITAL LETTER G,Lu,1,⠠⠛,0.235750,0.235750



EXCLUDED SOURCE CHARACTERS



,representation_system,character,codepoint,unicode,unicode_name,category,count,reason
0,International Morse Code,;,59,U+003B,SEMICOLON,Po,4,no standardized Morse representation



Representation integrity checks passed.


In [70]:
# ============================================================
# MATCHED ENGLISH FONT vs MORSE / UEB COMPARISONS
# ============================================================
#
# Purpose:
#
# Compare English rendered in ordinary fonts with English
# encoded in Morse or UEB using EXACTLY THE SAME source-
# character population on both sides of each comparison.
#
# Example:
#
# If ";" has no standardized Morse representation:
#
#     ";" is excluded from Morse
#     ";" is ALSO excluded from the English-font comparator
#
# But ";" may remain in the UEB comparison if UEB supports it.
#
# Therefore the Morse-matched and UEB-matched English font
# populations are allowed to differ.
#
# The original general English font complexity calculations
# are NOT modified.
# ============================================================


matched_comparison_records = []

matched_font_character_records = []


if run_english_representation_analysis:

    # --------------------------------------------------------
    # English font character-level data
    # --------------------------------------------------------

    english_font_characters = (
        character_complexity[
            character_complexity[
                "language"
            ] == "eng"
        ]
        .copy()
    )


    if english_font_characters.empty:

        raise ValueError(
            "No English font character-complexity records "
            "were found."
        )


    # --------------------------------------------------------
    # Analyze each representation system separately
    # --------------------------------------------------------

    matched_systems = [
        "International Morse Code",
        "Unified English Braille"
    ]


    for system in matched_systems:

        print()
        print("=" * 70)
        print(
            f"MATCHED COMPARISON: ENGLISH FONT vs {system}"
        )
        print("=" * 70)


        # ----------------------------------------------------
        # Get the source-character population actually used
        # by this representation system.
        # ----------------------------------------------------

        representation_characters = (
            representation_character_complexity[
                representation_character_complexity[
                    "representation_system"
                ] == system
            ]
            .copy()
        )


        if representation_characters.empty:

            raise ValueError(
                f"No character-level records found for "
                f"{system}."
            )


        supported_source_codepoints = set(
            representation_characters[
                "codepoint"
            ].astype(int)
        )


        # ----------------------------------------------------
        # Restrict English font records to exactly those
        # source characters.
        # ----------------------------------------------------

        matched_font_characters = (
            english_font_characters[
                english_font_characters[
                    "codepoint"
                ]
                .astype(int)
                .isin(
                    supported_source_codepoints
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Determine expected source-character count
        # ----------------------------------------------------

        representation_source_counts = (
            representation_characters[
                [
                    "codepoint",
                    "count"
                ]
            ]
            .drop_duplicates(
                subset=[
                    "codepoint"
                ]
            )
        )


        expected_character_count = int(
            representation_source_counts[
                "count"
            ].sum()
        )


        expected_unique_characters = int(
            representation_source_counts[
                "codepoint"
            ].nunique()
        )


        # ----------------------------------------------------
        # Calculate matched English complexity separately
        # for every font.
        # ----------------------------------------------------

        for (
            font_id,
            font_group
        ) in matched_font_characters.groupby(
            "font_id"
        ):

            font_group = (
                font_group.copy()
            )


            font_name = (
                font_group[
                    "font_name"
                ].iloc[0]
            )


            # -----------------------------------------------
            # Verify this font has every character required
            # for this particular matched comparison.
            # -----------------------------------------------

            font_codepoints = set(
                font_group[
                    "codepoint"
                ].astype(int)
            )


            missing_codepoints = (
                supported_source_codepoints
                - font_codepoints
            )


            if missing_codepoints:

                print()
                print(
                    f"Skipping {font_name}: "
                    f"{len(missing_codepoints):,} matched "
                    "source characters are unavailable."
                )

                continue


            # -----------------------------------------------
            # Verify source-character counts
            # -----------------------------------------------

            matched_character_count = int(
                font_group[
                    "count"
                ].sum()
            )


            matched_unique_characters = int(
                font_group[
                    "codepoint"
                ].nunique()
            )


            if (
                matched_character_count
                != expected_character_count
            ):

                raise ValueError(
                    f"{font_name} / {system}: matched "
                    "source-character count does not equal "
                    "the representation source-character "
                    "count."
                )


            if (
                matched_unique_characters
                != expected_unique_characters
            ):

                raise ValueError(
                    f"{font_name} / {system}: matched "
                    "unique-character count does not equal "
                    "the representation unique-character "
                    "count."
                )


            # -----------------------------------------------
            # Calculate matched English font complexity
            # -----------------------------------------------

            matched_font_total_complexity = float(
                font_group[
                    "weighted_complexity"
                ].sum()
            )


            matched_font_normalized_complexity = (
                matched_font_total_complexity
                / matched_character_count
                if matched_character_count > 0
                else np.nan
            )


            # -----------------------------------------------
            # Representation complexity
            # -----------------------------------------------

            representation_total_complexity = float(
                representation_characters[
                    "weighted_complexity"
                ].sum()
            )


            representation_normalized_complexity = (
                representation_total_complexity
                / expected_character_count
                if expected_character_count > 0
                else np.nan
            )


            # -----------------------------------------------
            # Store summary comparison
            # -----------------------------------------------

            matched_comparison_records.append(
                {
                    "language":
                        "eng",

                    "language_name":
                        language_names.get(
                            "eng",
                            "English"
                        ),

                    "font_id":
                        font_id,

                    "font_name":
                        font_name,

                    "representation_system":
                        system,

                    "matched_characters":
                        matched_character_count,

                    "matched_unique_characters":
                        matched_unique_characters,

                    "font_total_complexity":
                        matched_font_total_complexity,

                    "font_normalized_complexity":
                        matched_font_normalized_complexity,

                    "representation_total_complexity":
                        representation_total_complexity,

                    "representation_normalized_complexity":
                        representation_normalized_complexity
                }
            )


            # -----------------------------------------------
            # Store matched character-level font records
            # -----------------------------------------------

            for _, char_row in font_group.iterrows():

                matched_font_character_records.append(
                    {
                        "language":
                            "eng",

                        "language_name":
                            language_names.get(
                                "eng",
                                "English"
                            ),

                        "comparison_system":
                            system,

                        "font_id":
                            font_id,

                        "font_name":
                            font_name,

                        "character":
                            char_row[
                                "character"
                            ],

                        "codepoint":
                            int(
                                char_row[
                                    "codepoint"
                                ]
                            ),

                        "unicode":
                            char_row[
                                "unicode"
                            ],

                        "unicode_name":
                            char_row[
                                "unicode_name"
                            ],

                        "category":
                            char_row[
                                "category"
                            ],

                        "count":
                            int(
                                char_row[
                                    "count"
                                ]
                            ),

                        "complexity":
                            float(
                                char_row[
                                    "complexity"
                                ]
                            ),

                        "weighted_complexity":
                            float(
                                char_row[
                                    "weighted_complexity"
                                ]
                            )
                    }
                )


        print()

        print(
            f"Matched source-character occurrences: "
            f"{expected_character_count:,}"
        )

        print(
            f"Matched unique source characters: "
            f"{expected_unique_characters:,}"
        )


    # --------------------------------------------------------
    # Create final matched datasets
    # --------------------------------------------------------

    matched_representation_comparison = pd.DataFrame(
        matched_comparison_records
    )


    matched_font_character_complexity = pd.DataFrame(
        matched_font_character_records
    )


    # --------------------------------------------------------
    # Final integrity checks
    # --------------------------------------------------------

    if matched_representation_comparison.empty:

        raise ValueError(
            "No matched font/representation comparisons "
            "were produced."
        )


    if (
        matched_representation_comparison[
            [
                "font_normalized_complexity",
                "representation_normalized_complexity"
            ]
        ]
        .isna()
        .any()
        .any()
    ):

        raise ValueError(
            "Matched comparison contains missing "
            "complexity values."
        )


    print()
    print("=" * 70)
    print("MATCHED REPRESENTATION COMPARISONS")
    print("=" * 70)
    print()

    display(
        matched_representation_comparison
        .sort_values(
            [
                "representation_system",
                "font_name"
            ]
        )
        .reset_index(drop=True)
    )


    print()
    print(
        "Matched comparison integrity checks passed."
    )


else:

    matched_representation_comparison = (
        pd.DataFrame()
    )

    matched_font_character_complexity = (
        pd.DataFrame()
    )


MATCHED COMPARISON: ENGLISH FONT vs International Morse Code

Matched source-character occurrences: 8,948
Matched unique source characters: 55

MATCHED COMPARISON: ENGLISH FONT vs Unified English Braille

Matched source-character occurrences: 8,952
Matched unique source characters: 56

MATCHED REPRESENTATION COMPARISONS



,language,language_name,font_id,font_name,representation_system,matched_characters,matched_unique_characters,font_total_complexity,font_normalized_complexity,representation_total_complexity,representation_normalized_complexity
0,eng,English,2142,Noto Sans CJK SC,International Morse Code,8948,55,1683.095925,0.188097,1358.122403,0.151779
1,eng,English,2288,Noto Sans Mono Condensed Black,International Morse Code,8948,55,1630.934787,0.182268,1358.122403,0.151779
2,eng,English,1344,Noto Sans Mono Condensed Light,International Morse Code,8948,55,1685.307740,0.188345,1358.122403,0.151779
3,eng,English,2213,Noto Sans Mono ExtraCondensed SemiBold,International Morse Code,8948,55,1645.604387,0.183908,1358.122403,0.151779
4,eng,English,2211,Noto Sans Mono ExtraCondensed Thin,International Morse Code,8948,55,1744.560067,0.194966,1358.122403,0.151779
5,eng,English,2142,Noto Sans CJK SC,Unified English Braille,8952,56,1683.853083,0.188098,1711.817488,0.191222
6,eng,English,2288,Noto Sans Mono Condensed Black,Unified English Braille,8952,56,1631.635574,0.182265,1711.817488,0.191222
7,eng,English,1344,Noto Sans Mono Condensed Light,Unified English Braille,8952,56,1686.036410,0.188342,1711.817488,0.191222
8,eng,English,2213,Noto Sans Mono ExtraCondensed SemiBold,Unified English Braille,8952,56,1646.302746,0.183903,1711.817488,0.191222
9,eng,English,2211,Noto Sans Mono ExtraCondensed Thin,Unified English Braille,8952,56,1745.300866,0.194962,1711.817488,0.191222



Matched comparison integrity checks passed.


In [75]:
# ============================================================
# FINAL TEXT / CHARACTER RECONSTRUCTION CHECK
# ============================================================

reconstructed = (
    character_complexity
    .groupby(
        [
            "language",
            "font_id"
        ],
        as_index=False
    )
    .agg(
        reconstructed_total_complexity=(
            "weighted_complexity",
            "sum"
        ),
        reconstructed_nonwhitespace_characters=(
            "count",
            "sum"
        )
    )
)


reconstructed[
    "reconstructed_normalized_complexity"
] = (
    reconstructed[
        "reconstructed_total_complexity"
    ]
    / reconstructed[
        "reconstructed_nonwhitespace_characters"
    ]
)


check = (
    text_complexity_table.merge(
        reconstructed,
        on=[
            "language",
            "font_id"
        ],
        how="left",
        validate="one_to_one"
    )
)


if not np.allclose(
    check[
        "total_complexity"
    ],
    check[
        "reconstructed_total_complexity"
    ]
):

    raise ValueError(
        "Character-level records do not reconstruct "
        "text-level total complexity."
    )


if not np.allclose(
    check[
        "normalized_complexity"
    ],
    check[
        "reconstructed_normalized_complexity"
    ]
):

    raise ValueError(
        "Character-level records do not reconstruct "
        "text-level normalized complexity."
    )


print(
    "Final text/character reconstruction check passed."
)

Final text/character reconstruction check passed.


In [74]:
# ============================================================
# EXPORT COMPLETE TEXT COMPLEXITY DATASETS
# ============================================================
#
# Four CSV datasets are produced:
#
# 1. text_complexity_by_font.csv
#       One row per text × font.
#       General text-complexity results for all languages.
#
# 2. character_complexity_by_text_font.csv
#       One row per source character × text × font.
#
# 3. english_representation_comparison.csv
#       English fonts, Morse, and UEB in ONE comparison table.
#
#       English font values are recalculated separately for
#       each representation system so that both sides use
#       exactly the same source-character population.
#
# 4. english_representation_character_complexity.csv
#       Character-level records underlying the matched English
#       font, Morse, and UEB comparisons.
#
# The ordinary text/font datasets remain unchanged.
# ============================================================


# ------------------------------------------------------------
# GENERAL FONT TEXT-LEVEL DATA
# ------------------------------------------------------------

text_complexity_csv_path = os.path.join(
    output_dir,
    "text_complexity_by_font.csv"
)


text_complexity_table.to_csv(
    text_complexity_csv_path,
    index=False,
    encoding="utf-8"
)


print()
print("TEXT × FONT COMPLEXITY CSV SAVED")
print(text_complexity_csv_path)

print(
    f"Rows: "
    f"{len(text_complexity_table):,}"
)

print(
    f"Languages: "
    f"{text_complexity_table['language'].nunique():,}"
)

print(
    f"Fonts: "
    f"{text_complexity_table['font_id'].nunique():,}"
)


# ------------------------------------------------------------
# GENERAL FONT CHARACTER-LEVEL DATA
# ------------------------------------------------------------

character_complexity_csv_path = os.path.join(
    output_dir,
    "character_complexity_by_text_font.csv"
)


character_complexity.to_csv(
    character_complexity_csv_path,
    index=False,
    encoding="utf-8"
)


print()
print("CHARACTER × TEXT × FONT COMPLEXITY CSV SAVED")
print(character_complexity_csv_path)

print(
    f"Rows: "
    f"{len(character_complexity):,}"
)


# ============================================================
# BUILD UNIFIED MATCHED ENGLISH REPRESENTATION TABLE
# ============================================================

english_comparison_records = []

english_comparison_character_records = []


if (
    run_english_representation_analysis
    and not matched_representation_comparison.empty
):

    # --------------------------------------------------------
    # Add matched English FONT rows
    # --------------------------------------------------------

    for _, row in matched_representation_comparison.iterrows():

        comparison_system = (
            row["representation_system"]
        )


        english_comparison_records.append(
            {
                "language":
                    "eng",

                "language_name":
                    row["language_name"],

                "comparison_system":
                    comparison_system,

                "representation_type":
                    "Font",

                "representation_name":
                    row["font_name"],

                "font_id":
                    row["font_id"],

                "normalized_complexity":
                    row[
                        "font_normalized_complexity"
                    ],

                "total_complexity":
                    row[
                        "font_total_complexity"
                    ],

                "matched_characters":
                    row[
                        "matched_characters"
                    ],

                "matched_unique_characters":
                    row[
                        "matched_unique_characters"
                    ]
            }
        )


    # --------------------------------------------------------
    # Add one Morse / UEB row per matched comparison population
    # --------------------------------------------------------

    for system in [
        "International Morse Code",
        "Unified English Braille"
    ]:

        system_rows = (
            matched_representation_comparison[
                matched_representation_comparison[
                    "representation_system"
                ] == system
            ]
        )


        if system_rows.empty:

            continue


        # Representation values are repeated for every font
        # in matched_representation_comparison, so use the
        # first row after verifying consistency.

        normalized_values = (
            system_rows[
                "representation_normalized_complexity"
            ]
            .dropna()
            .unique()
        )


        total_values = (
            system_rows[
                "representation_total_complexity"
            ]
            .dropna()
            .unique()
        )


        matched_counts = (
            system_rows[
                "matched_characters"
            ]
            .dropna()
            .unique()
        )


        matched_unique_counts = (
            system_rows[
                "matched_unique_characters"
            ]
            .dropna()
            .unique()
        )


        if len(normalized_values) != 1:

            raise ValueError(
                f"{system}: inconsistent normalized "
                "complexity values across matched rows."
            )


        if len(total_values) != 1:

            raise ValueError(
                f"{system}: inconsistent total complexity "
                "values across matched rows."
            )


        if len(matched_counts) != 1:

            raise ValueError(
                f"{system}: inconsistent matched "
                "character counts."
            )


        if len(matched_unique_counts) != 1:

            raise ValueError(
                f"{system}: inconsistent matched unique "
                "character counts."
            )


        english_comparison_records.append(
            {
                "language":
                    "eng",

                "language_name":
                    language_names.get(
                        "eng",
                        "English"
                    ),

                "comparison_system":
                    system,

                "representation_type":
                    "Encoded representation",

                "representation_name":
                    system,

                "font_id":
                    np.nan,

                "normalized_complexity":
                    float(
                        normalized_values[0]
                    ),

                "total_complexity":
                    float(
                        total_values[0]
                    ),

                "matched_characters":
                    int(
                        matched_counts[0]
                    ),

                "matched_unique_characters":
                    int(
                        matched_unique_counts[0]
                    )
            }
        )


    # --------------------------------------------------------
    # Character-level matched FONT records
    # --------------------------------------------------------

    for _, row in matched_font_character_complexity.iterrows():

        english_comparison_character_records.append(
            {
                "language":
                    "eng",

                "language_name":
                    row["language_name"],

                "comparison_system":
                    row["comparison_system"],

                "representation_type":
                    "Font",

                "representation_name":
                    row["font_name"],

                "font_id":
                    row["font_id"],

                "character":
                    row["character"],

                "codepoint":
                    row["codepoint"],

                "unicode":
                    row["unicode"],

                "unicode_name":
                    row["unicode_name"],

                "category":
                    row["category"],

                "count":
                    row["count"],

                "representation":
                    row["character"],

                "complexity":
                    row["complexity"],

                "weighted_complexity":
                    row["weighted_complexity"]
            }
        )


    # --------------------------------------------------------
    # Character-level Morse / UEB records
    # --------------------------------------------------------

    for _, row in representation_character_complexity.iterrows():

        english_comparison_character_records.append(
            {
                "language":
                    "eng",

                "language_name":
                    row["language_name"],

                "comparison_system":
                    row[
                        "representation_system"
                    ],

                "representation_type":
                    "Encoded representation",

                "representation_name":
                    row[
                        "representation_system"
                    ],

                "font_id":
                    np.nan,

                "character":
                    row["character"],

                "codepoint":
                    row["codepoint"],

                "unicode":
                    row["unicode"],

                "unicode_name":
                    row["unicode_name"],

                "category":
                    row["category"],

                "count":
                    row["count"],

                "representation":
                    row["representation"],

                "complexity":
                    row["complexity"],

                "weighted_complexity":
                    row["weighted_complexity"]
            }
        )


# ============================================================
# CREATE + VERIFY UNIFIED ENGLISH DATASETS
# ============================================================

english_representation_comparison = pd.DataFrame(
    english_comparison_records
)


english_representation_character_complexity = pd.DataFrame(
    english_comparison_character_records
)


if not english_representation_comparison.empty:

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    for system in [
        "International Morse Code",
        "Unified English Braille"
    ]:

        system_rows = (
            english_representation_comparison[
                english_representation_comparison[
                    "comparison_system"
                ] == system
            ]
        )


        if system_rows.empty:

            raise ValueError(
                f"No unified comparison rows found for "
                f"{system}."
            )


        matched_counts = (
            system_rows[
                "matched_characters"
            ]
            .unique()
        )


        if len(matched_counts) != 1:

            raise ValueError(
                f"{system}: unified comparison does not "
                "use one matched source-character count."
            )


        representation_rows = (
            system_rows[
                system_rows[
                    "representation_type"
                ] == "Encoded representation"
            ]
        )


        if len(representation_rows) != 1:

            raise ValueError(
                f"{system}: expected exactly one encoded "
                "representation summary row."
            )


    # --------------------------------------------------------
    # Save unified text-level comparison
    # --------------------------------------------------------

    english_comparison_csv_path = os.path.join(
        output_dir,
        "english_representation_comparison.csv"
    )


    english_representation_comparison.to_csv(
        english_comparison_csv_path,
        index=False,
        encoding="utf-8"
    )


    # --------------------------------------------------------
    # Save unified character-level comparison
    # --------------------------------------------------------

    english_comparison_character_csv_path = os.path.join(
        output_dir,
        "english_representation_character_complexity.csv"
    )


    english_representation_character_complexity.to_csv(
        english_comparison_character_csv_path,
        index=False,
        encoding="utf-8"
    )


    print()
    print("=" * 70)
    print("MATCHED ENGLISH REPRESENTATION DATASETS SAVED")
    print("=" * 70)

    print()
    print("TEXT-LEVEL COMPARISON:")
    print(english_comparison_csv_path)

    print(
        f"Rows: "
        f"{len(english_representation_comparison):,}"
    )

    print()
    print("CHARACTER-LEVEL COMPARISON:")
    print(english_comparison_character_csv_path)

    print(
        f"Rows: "
        f"{len(english_representation_character_complexity):,}"
    )


else:

    print()
    print(
        "Matched English representation CSVs not created "
        "because English representation analysis was not "
        "available."
    )


TEXT × FONT COMPLEXITY CSV SAVED
/content/drive/MyDrive/Character Complexity/text_complexity_by_font.csv
Rows: 1,123
Languages: 236
Fonts: 5

CHARACTER × TEXT × FONT COMPLEXITY CSV SAVED
/content/drive/MyDrive/Character Complexity/character_complexity_by_text_font.csv
Rows: 67,895

MATCHED ENGLISH REPRESENTATION DATASETS SAVED

TEXT-LEVEL COMPARISON:
/content/drive/MyDrive/Character Complexity/english_representation_comparison.csv
Rows: 12

CHARACTER-LEVEL COMPARISON:
/content/drive/MyDrive/Character Complexity/english_representation_character_complexity.csv
Rows: 666


## CSV Output Guide

### `text_complexity_by_font.csv`
Contains the **overall complexity score for each text in each usable font**.

Use this file to compare the visual complexity of different languages or writing systems.

- One row = one text × one font
- `normalized_complexity` = main complexity score to use for comparisons
- `total_complexity` = total complexity before correcting for text length

### `character_complexity_by_text_font.csv`
Contains the **individual character data used to calculate the text complexity scores**.

Use this file to examine which characters contribute to a text's complexity.

- One row = one character × one text × one font
- `count` = how often the character occurs
- `complexity` = visual complexity of that character
- `weighted_complexity` = `count × complexity`

### `english_representation_comparison.csv`
Contains the **English font, Morse Code, and Braille text-level complexity scores**.

Use this file to compare ordinary written English with Morse and Braille.

Morse and Braille are compared separately with English. If a character is not supported by Morse or Braille, it is removed from both that representation and its matching English calculation so the comparison uses the same source characters.

### `english_representation_character_complexity.csv`
Contains the **character-level data behind the English, Morse, and Braille comparison**.

Use this file to see how individual English characters and their Morse or Braille representations contribute to the overall complexity scores.

- One row = one source character in one representation
- Includes character frequency, complexity, and weighted complexity
- Morse sequences and multi-cell Braille representations are treated as representations of a single source character